In [ ]:
from netCDF4 import Dataset, MFDataset
import numpy as np
import numpy.ma as ma
import matplotlib.pyplot as plt
import xarray as xr 
from mpl_toolkits.basemap import Basemap
from mpl_toolkits.basemap import shiftgrid
from cartopy.util import add_cyclic_point
import matplotlib
import random
from time import perf_counter
import pickle
from sklearn.linear_model import LinearRegression

plt.rc('text.latex', preamble=r'\usepackage[cm]{sfmath}')
#matplotlib.rcParams['mathtext.fontset'] = 'cm'
#matplotlib.rcParams['mathtext.rm'] = 'Bitstream Vera Sans'
#matplotlib.rcParams['mathtext.it'] = 'Bitstream Vera Sans:italic'
#matplotlib.rcParams['mathtext.bf'] = 'Bitstream Vera Sans:bold'
#matplotlib.rcParams['font.family'] = 'sans-serif'
import matplotlib.gridspec as gridspec
plt.rc('text', usetex=True)
#plt.rc('font', family='sans-serif')

plt.rc('text.latex', preamble=r'\usepackage[cm]{sfmath}')
plt.rcParams.update({'font.size': 12})

# Global variables and functions

In [ ]:
names = ["CESM2-WACCM", "CNRM-ESM2-1", "GFDL-ESM4", "GISS-E2-1-G", "MRI-ESM2-0", "UKESM1-0-LL"]

In [ ]:
def meanvar(dataset, var, p=None, lat_low=-90, lat_high=90):
    '''
    Calculates the spatial mean of the variable var on pressure level p between the latidues lat_low and lat_high for each year
    '''
    if var=="tas":
        if "TREFHT" in list(dataset.keys()):
            dat = dataset["TREFHT"]
        elif "T" in list(dataset.keys()):
            dat = dataset["T"].sel(plev=10**5)
        else:
            dat = dataset["tas"]
    elif var==None:
        dat = dataset
    else:
        dat = dataset[var]
    if p != None:
        dat = dat.sel(plev=p)
    # var_t = var.sel(time=slice(str(tstart), str(tend)))
    if "lon" in dataset.dims:
        dat = dat.mean(dim="lon")
    elif "longitude" in dataset.dims:
        dat = dat.mean(dim="longitude")
    if "lat" in dataset.dims:
        dat_slice = dat.sel(lat=slice(lat_low, lat_high))
        weights = np.cos(np.deg2rad(dat_slice["lat"]))
        dat_weighted = dat_slice.weighted(weights)
        dat_area_mean = dat_weighted.mean(dim="lat")
    elif "latitude" in dataset.dims:
        dat_slice = dat.sel(latitude=slice(lat_low, lat_high))
        weights = np.cos(np.deg2rad(dat_slice["latitude"]))
        dat_weighted = dat_slice.weighted(weights)
        dat_area_mean = dat_weighted.mean(dim="latitude")
    if "time" in dataset.dims:
        dat_mean = dat_area_mean.groupby("time.year").mean() 
    else:
        dat_mean = dat_area_mean
    return dat_mean

In [ ]:
def prepdato3(data, start, end):
    if "o3" in data.data_vars:
        d = data["o3"]
    elif "O3" in data.data_vars:
        d = data["O3"]
    else:
        raise KeyError("No viable data variable found")
    
    dsel = d.sel(time=slice(str(start), str(end)))
    dsel_mean = dsel.mean(dim="time")
    if "lon" in data.dims:
        dsel_mean = dsel_mean.mean(dim="lon")
    
    return dsel_mean

def prepdato3_cesm(data, start, end):
    if "o3" in data.data_vars:
        d = data["o3"]
    elif "O3" in data.data_vars:
        d = data["O3"]
    else:
        raise KeyError("No viable data variable found")
    
    start_idx = (start - 1950)*12
    end_idx = (end - 1950 + 1)*12
    dsel_mean = d[start_idx:end_idx].mean(dim="time")
    if "lon" in data.dims:
        dsel_mean = dsel_mean.mean(dim="lon")
    
    return dsel_mean

def prepdato3_ncc(data, start, end):
    if "o3" in data.data_vars:
        d = data["o3"]
    elif "O3" in data.data_vars:
        d = data["O3"]
    else:
        raise KeyError("No viable data variable found")
    
    start_idx = (start - 1955)*12
    end_idx = (end - 1955 + 1)*12
    dsel_mean = d[start_idx:end_idx].mean(dim="time")
    if "lon" in data.dims:
        dsel_mean = dsel_mean.mean(dim="lon")
    
    return dsel_mean

def latmean(var):
    
    if 'lat' in var.dims:
        if var.lat[0]<var.lat[1]: var=var.sel(lat=slice(-90,90))
        else: var=var.sel(lat=slice(90,-90))
        weights = np.cos(np.deg2rad(var.lat))
        var = var.weighted(weights)     
        var=var.mean(dim='lat')
    if 'latitude' in var.dims:
        if var.latitude[0]<var.latitude[1]: var=var.sel(latitude=slice(-90,90))
        else: var=var.sel(latitude=slice(90,-90))

        weights = np.cos(np.deg2rad(var.latitude))
        var = var.weighted(weights)     
        var=var.mean(dim='latitude')
    
    if 'lon' in var.dims: var=var.mean(dim='lon')
    if 'longitude' in var.dims: var=var.mean(dim='longitude')
    
    return var

In [ ]:
def meanvar_cesm(dataset, var="tas", p=None, lat_low=-90, lat_high=90):
    '''
    Calculates the spatial mean of the variable var on pressure level p between the latidues lat_low and lat_high for each year
    Adopted for new CESM data
    '''
    if var=="tas":
        if "TREFHT" in list(dataset.keys()):
            dat = dataset["TREFHT"]
        else:
            dat = dataset["tas"]
    else:
        dat = dataset[var]
    if p != None:
        dat = dat.sel(plev=p)
    # var_t = var.sel(time=slice(str(tstart), str(tend)))
    if "lon" in dataset.dims:
        dat = dat.mean(dim="lon")
    dat_slice = dat.sel(lat=slice(lat_low, lat_high))
    weights = np.cos(np.deg2rad(dat_slice["lat"]))
    dat_weighted = dat_slice.weighted(weights)
    dat_area_mean = dat_weighted.mean(dim="lat")
    dat_mean = np.zeros(len(dat_area_mean) // 12)
    for i in range(len(dat_mean)):
        dat_mean[i] = dat_area_mean[12*i:12*(i+1)].mean()
    return dat_mean

def warming_indiv(data_list, ll=-90, lh=90):
    '''Calculates the temperature trend for each element of data_list individually'''
    res_list = list()
    for i in range(len(data_list)):
        helper = meanvar(data_list[i], var="tas", lat_low = ll, lat_high=lh)
        res_list.append((helper.sel(year=slice("1996", "2005")).mean() - helper.sel(year=slice("1955", "1964")).mean()).item())
    
    return res_list

def warming_all(data_list, ll=-90, lh=90):
    '''Calculates the temperature trend for each element of data_list individually'''

    list_all=np.empty((10, len(data_list)))
    list_all_early=np.empty((10, len(data_list)))
    for i in range(len(data_list)):
        helper = meanvar(data_list[i], var="tas", lat_low = ll, lat_high=lh)
        list_all[:,i]=helper.sel(year=slice("1996", "2005"))
        list_all_early[:,i]=helper.sel(year=slice("1955", "1964"))
        
    mean_warming= np.mean(list_all)-np.mean(list_all_early)
    mean_warming_std=np.sqrt(np.var(list_all) + np.var(list_all_early))
    
    return mean_warming, mean_warming_std

In [ ]:
def calc_parc_o3(var):
    
        m_air = 28.964/(6.022E23)
        g = 980.6
    
        if 'plev' in var.dims: 
            plev=var.plev
            level='plev'
        if 'lev' in var.dims: 
            plev=var.lev
            level='lev'
        if 'level' in var.dims: 
            plev=var.level
            level='level'
            
        if 'lv_ISBL1' in var.dims:
            plev=var.lv_ISBL1
            level='lv_ISBL1'
        
     #   time=var.time
        delta_p = np.zeros((len(plev)))
        
        if plev[0]<plev[len(plev)-1] and plev[len(plev)-1] <= 1000: 
            factor=100
            factor_2=1 # conversion Pa->hPa
        if plev[0]>plev[len(plev)-1] and plev[0] <=1000: 
            factor=100
            factor_2=1
        if plev[0]<plev[len(plev)-1] and plev[len(plev)-1] >1000: 
            factor=1
            factor_2=100
        if plev[0]>plev[len(plev)-1] and plev[0] >1000: 
            factor=1
            factor_2=100
            
       
        if plev[0]<plev[len(plev)-1]: # for pressure levels in hPa
            
            for levelx in range(1,len(plev)): 
                delta_p[levelx]=float(plev[levelx] - plev[levelx-1])
               
            weights_p = xr.DataArray(delta_p*factor, dims=[level], coords=[plev]) # difference between pressure levels in Pa
 
 
            O3 = var * weights_p * 10/ (g * m_air)
    
        
            if level=='plev': O3=O3.sel(plev=slice(30*factor_2,100*factor_2)) 
            if level=='lev': O3=O3.sel(lev=slice(30*factor_2,100*factor_2))
            if level=='level': O3=O3.sel(level=slice(30*factor_2,100*factor_2))

            O3 = O3.sum(dim=level)
        
            O3 = O3/2.687E16  #calculate DU
            
            
        if plev[0]>plev[len(plev)-1]: # for pressure levels in hPa
            
            for levelx in range(0,len(plev)-1): 
                delta_p[levelx]= float(plev[levelx] - plev[levelx+1])

            weights_p = xr.DataArray(delta_p*factor, dims=[level], coords=[plev]) # difference between pressure levels in Pa

            O3 = var * weights_p * 10/ (g * m_air)
       
            
            if level =='plev': O3=O3.sel(plev=slice(100*factor_2,30*factor_2)) 
            if level=='lev': O3=O3.sel(lev=slice(100*factor_2,30*factor_2)) 
            if level=='level': O3=O3.sel(level=slice(100*factor_2,30*factor_2)) 
                
            O3 = O3.sum(dim=level)
            O3 = O3/2.687E16  #calculate DU
    
        return O3 #.where(O3 != 0)

# Data Import

## Radiation

Import Data from piClim simulation to calculate ERF

rlut: TOA Outgoing Longwave Radiation

rsdt: TOA Incident Shortwave Radiation

rsut: TOA Outgoing Shortwave Radiation

0: CESM / 1: CNRM / 2: GFDL / 3: GISS / 4: MRI / 5: UKESM

### Import coupled HC simulations:

In [ ]:
rlut_control = list()

rlut_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/CESM2-WACCM/rlut_Amon_CESM2-WACCM_piClim-control_r1i2p1f1_gn_000101-003012.nc"))
rlut_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/CNRM-ESM2-1/rlut_Amon_CNRM-ESM2-1_piClim-control_r1i1p1f2_gr_185001-187912.nc"))
rlut_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/GFDL-ESM4/rlut_Amon_GFDL-ESM4_piClim-control_r1i1p1f1_gr1_000101-003012.nc"))
rlut_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/GISS-E2-1-G/rlut_Amon_GISS-E2-1-G_piClim-control_r1i1p3f1_gn_195001-199012.nc"))
rlut_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/MRI-ESM2-0/rlut_Amon_MRI-ESM2-0_piClim-control_r1i1p1f1_gn_185001-188012.nc"))
rlut_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/UKESM1-0-LL/rlut_Amon_UKESM1-0-LL_piClim-control_r1i1p1f4_gn_185001-189412.nc"))

In [ ]:
rsdt_control = list()

rsdt_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/CESM2-WACCM/rsdt_Amon_CESM2-WACCM_piClim-control_r1i2p1f1_gn_000101-003012.nc"))
rsdt_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/CNRM-ESM2-1/rsdt_Amon_CNRM-ESM2-1_piClim-control_r1i1p1f2_gr_185001-187912.nc"))
rsdt_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/GFDL-ESM4/rsdt_Amon_GFDL-ESM4_piClim-control_r1i1p1f1_gr1_000101-003012.nc"))
rsdt_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/GISS-E2-1-G/rsdt_Amon_GISS-E2-1-G_piClim-control_r1i1p3f1_gn_195001-199012.nc"))
rsdt_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/MRI-ESM2-0/rsdt_Amon_MRI-ESM2-0_piClim-control_r1i1p1f1_gn_185001-188012.nc"))
rsdt_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/UKESM1-0-LL/rsdt_Amon_UKESM1-0-LL_piClim-control_r1i1p1f4_gn_185001-189412.nc"))

In [ ]:
rsut_control = list()

rsut_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/CESM2-WACCM/rsut_Amon_CESM2-WACCM_piClim-control_r1i2p1f1_gn_000101-003012.nc"))
rsut_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/CNRM-ESM2-1/rsut_Amon_CNRM-ESM2-1_piClim-control_r1i1p1f2_gr_185001-187912.nc"))
rsut_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/GFDL-ESM4/rsut_Amon_GFDL-ESM4_piClim-control_r1i1p1f1_gr1_000101-003012.nc"))
rsut_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/GISS-E2-1-G/rsut_Amon_GISS-E2-1-G_piClim-control_r1i1p3f1_gn_195001-199012.nc"))
rsut_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/MRI-ESM2-0/rsut_Amon_MRI-ESM2-0_piClim-control_r1i1p1f1_gn_185001-188012.nc"))
rsut_control.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/UKESM1-0-LL/rsut_Amon_UKESM1-0-LL_piClim-control_r1i1p1f4_gn_185001-189412.nc"))

In [ ]:
with open('piclim_control_rad', 'wb') as f:
    pickle.dump([rlut_control, rsdt_control, rsut_control], f)

In [ ]:
with open('piclim_control_rad', 'rb') as f:
    [rlut_control, rsdt_control, rsut_control] = pickle.load(f)

### Import fixed HC simulations:

In [ ]:
rlut_hc = list()

rlut_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/CESM2-WACCM/rlut_Amon_CESM2-WACCM_piClim-HC_r1i1p1f1_gn_000101-003112.nc"))
rlut_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/CNRM-ESM2-1/rlut_Amon_CNRM-ESM2-1_piClim-HC_r1i1p1f2_gr_185001-187912.nc"))
rlut_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/GFDL-ESM4/rlut_Amon_GFDL-ESM4_piClim-HC_r1i1p1f1_gr1_000101-003012.nc"))
rlut_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/GISS-E2-1-G/rlut_Amon_GISS-E2-1-G_piClim-HC_r1i1p3f1_gn_195001-199012.nc"))
rlut_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/MRI-ESM2-0/rlut_Amon_MRI-ESM2-0_piClim-HC_r1i1p1f1_gn_185001-188012.nc"))
rlut_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/UKESM1-0-LL/rlut_Amon_UKESM1-0-LL_piClim-HC_r1i1p1f4_gn_185001-189412.nc"))

In [ ]:
rsdt_hc = list()

rsdt_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/CESM2-WACCM/rsdt_Amon_CESM2-WACCM_piClim-HC_r1i1p1f1_gn_000101-003112.nc"))
rsdt_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/CNRM-ESM2-1/rsdt_Amon_CNRM-ESM2-1_piClim-HC_r1i1p1f2_gr_185001-187912.nc"))
# File not available, taking the file from the control run as TOA Incident Shortwave Radiation is anyway the same
rsdt_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-control/GFDL-ESM4/rsdt_Amon_GFDL-ESM4_piClim-control_r1i1p1f1_gr1_000101-003012.nc"))

rsdt_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/GISS-E2-1-G/rsdt_Amon_GISS-E2-1-G_piClim-HC_r1i1p3f1_gn_195001-199012.nc"))
rsdt_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/MRI-ESM2-0/rsdt_Amon_MRI-ESM2-0_piClim-HC_r1i1p1f1_gn_185001-188012.nc"))
rsdt_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/UKESM1-0-LL/rsdt_Amon_UKESM1-0-LL_piClim-HC_r1i1p1f4_gn_185001-189412.nc"))

In [ ]:
rsut_hc = list()

rsut_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/CESM2-WACCM/rsut_Amon_CESM2-WACCM_piClim-HC_r1i1p1f1_gn_000101-003112.nc"))
rsut_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/CNRM-ESM2-1/rsut_Amon_CNRM-ESM2-1_piClim-HC_r1i1p1f2_gr_185001-187912.nc"))
rsut_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/GFDL-ESM4/rsut_Amon_GFDL-ESM4_piClim-HC_r1i1p1f1_gr1_000101-003012.nc"))
rsut_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/GISS-E2-1-G/rsut_Amon_GISS-E2-1-G_piClim-HC_r1i1p3f1_gn_195001-199012.nc"))
rsut_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/MRI-ESM2-0/rsut_Amon_MRI-ESM2-0_piClim-HC_r1i1p1f1_gn_185001-188012.nc"))
rsut_hc.append(xr.open_dataset("/net/ch4/chemie/wkonstantin/cimp6_data/piClim-HC/UKESM1-0-LL/rsut_Amon_UKESM1-0-LL_piClim-HC_r1i1p1f4_gn_185001-189412.nc"))

In [ ]:
with open('piclim_hc_rad', 'wb') as f:
    pickle.dump([rlut_hc, rsdt_hc, rsut_hc], f)

In [ ]:
with open('piclim_hc_rad', 'rb') as f:
    [rlut_hc, rsdt_hc, rsut_hc] = pickle.load(f)

## Ozone coupled runs
Note the shifted time frame (1958-1967 instead of 1955-1964), since JRA-55 is only available beginning in 1958

CESM:

In [ ]:
#import historical data

cesm_o3 = list()
for i in range(3):
    cesm_o3.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/o3_Amon/CESM2-WACCM/r' + str(i + 1)
                     + 'i1p1f1/gn/o3_Amon_CESM2-WACCM_historical_r' +  str(i + 1) + 'i1p1f1_gn_185001-201412.nc'))
    
for i in range(3):
    for j in range(4):
        cesm_o3.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/Additional_data/CESM2-WACCM/wa6_ic' + str(i + 1) + '.00' + str(j + 1) + 
                              '/wa6_ic' + str(i + 1) + '.00' + str(j + 1) + '.cam.h0.1950-2014.zm.nc'))

In [ ]:
# calculate mean O3 in early and late timeframe

cesm_o3t1 = list()
for i in range(len(cesm_o3)):
    if i in (0, 1, 2):
        cesm_o3t1.append(prepdato3(cesm_o3[i], start=1958, end=1967))
    else:
        cesm_o3t1.append(prepdato3_cesm(cesm_o3[i], start=1958, end=1967))
    
cesm_o3t2 = list()
for i in range(len(cesm_o3)):
    if i in (0, 1, 2):
        cesm_o3t2.append(prepdato3(cesm_o3[i], start=1996, end=2005))
    else:
        cesm_o3t2.append(prepdato3_cesm(cesm_o3[i], start=1996, end=2005))
    
cesm_o3t1_mean = sum(i for i in cesm_o3t1) / len(cesm_o3t1)
cesm_o3t2_mean = sum(i for i in cesm_o3t2) / len(cesm_o3t2)

In [ ]:
cesm_o3_histchange_100hpa_list = list() #calculate hist. O3 change in all ensemble members at 100 hPa

for i in range(len(cesm_o3)):
    cesm_o3_histchange_100hpa_list.append(float(meanvar(cesm_o3t2[i] - cesm_o3t1[i], None, p=10000)))

cesm_o3_histchange_100hpa = float(meanvar(cesm_o3t2_mean - cesm_o3t1_mean, None, p=10000))


cesm_o3_histchange_parcol_list=list() #calculate hist. O3 change in all ensemble members for PCO between 30 and 100 hPa

for i in range(len(cesm_o3)):
    
    cesm_o3t2_pc=calc_parc_o3(cesm_o3t2[i])
    cesm_o3t1_pc=calc_parc_o3(cesm_o3t1[i])
    
    cesm_o3_histchange_parcol_list.append(float(latmean(cesm_o3t2_pc) - latmean(cesm_o3t1_pc)))
    
cesm_o3_histchange_parcol = np.nanmean(cesm_o3_histchange_parcol_list)

In [ ]:
#import fixed HC data

cesm_o3_hc = list()
for i in range(1):
    cesm_o3_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/o3_Amon/CESM2-WACCM/r' + str(i + 1)
                     + 'i1p1f1/gn/o3_Amon_CESM2-WACCM_hist-1950HC_r' +  str(i + 1) + 'i1p1f1_gn_195001-201412.nc'))
    
for i in (2, 3, 4):
    cesm_o3_hc.append(xr.open_dataset('/net/reid/lhome/gchiodo/data/Columbia/wa6_ens.HC1950.00' + str(i) +
                                      '/wa6_ens.HC1950.00' +  str(i) + '.cam.h0.1950-2014.zm.nc'))

In [ ]:
# calculate mean O3 in early and late timeframe

cesm_o3_hct1 = list()
for i in range(len(cesm_o3_hc)):
    cesm_o3_hct1.append(prepdato3_cesm(cesm_o3_hc[i], start=1958, end=1967))
    
cesm_o3_hct2 = list()
for i in range(len(cesm_o3_hc)):
    cesm_o3_hct2.append(prepdato3_cesm(cesm_o3_hc[i], start=1996, end=2005))
    
cesm_o3_hct1_mean = sum(i for i in cesm_o3_hct1) / len(cesm_o3_hct1)
cesm_o3_hct2_mean = sum(i for i in cesm_o3_hct2) / len(cesm_o3_hct2)

In [ ]:
cesm_o3_hcchange_100hpa_list = list() #calculate hist. O3 change in all ensemble members at 100 hPa
for i in range(len(cesm_o3_hc)):
    cesm_o3_hcchange_100hpa_list.append(float(meanvar(cesm_o3_hct2[i] - cesm_o3_hct1[i], None, p=10000)))
    
cesm_o3_hcchange_100hpa = np.mean(cesm_o3_hcchange_100hpa_list)

cesm_o3_hcchange_parcol_list=list()

for i in range(len(cesm_o3_hc)): #calculate hist. O3 change in all ensemble members for PCO between 30 and 100 hPa
    
    cesm_o3_hct2_pc=calc_parc_o3(cesm_o3_hct2[i])
    cesm_o3_hct1_pc=calc_parc_o3(cesm_o3_hct1[i])
    
    cesm_o3_hcchange_parcol_list.append(float(latmean(cesm_o3_hct2_pc) - latmean(cesm_o3_hct1_pc)))

cesm_o3_hcchange_parcol = np.mean(cesm_o3_hcchange_parcol_list)

# calculate ozone changes due to HCs

cesm_o3_changeduetoHC_100hpa = [cesm_o3_histchange_100hpa - cesm_o3_hcchange_100hpa_list[i] for i in range(len(cesm_o3_hcchange_parcol_list))]
cesm_o3_changeduetoHC_parcol = [cesm_o3_histchange_parcol - cesm_o3_hcchange_parcol_list[i] for i in range(len(cesm_o3_hcchange_parcol_list))]

CNRM

In [ ]:
cnrm_o3 = list()

for i in range(11):
    cnrm_o3.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/o3_Amon/CNRM-ESM2-1/r' + str(i + 1)
                        + 'i1p1f2/gr/o3_Amon_CNRM-ESM2-1_historical_r' + str(i + 1) + 'i1p1f2_gr_185001-201412.nc'))

In [ ]:
cnrm_o3t1 = list()
for i in range(len(cnrm_o3)):
    cnrm_o3t1.append(prepdato3(cnrm_o3[i], start=1958, end=1967))
    
cnrm_o3t2 = list()
for i in range(len(cnrm_o3)):
    cnrm_o3t2.append(prepdato3(cnrm_o3[i], start=1996, end=2005))
    
cnrm_o3t1_mean = sum(i for i in cnrm_o3t1) / len(cnrm_o3t1)
cnrm_o3t2_mean = sum(i for i in cnrm_o3t2) / len(cnrm_o3t2)

In [ ]:
cnrm_o3_histchange_100hpa_list = list()
for i in range(len(cnrm_o3)):
    cnrm_o3_histchange_100hpa_list.append(float(meanvar(cnrm_o3t2[i] - cnrm_o3t1[i], None, p=10000)))

cnrm_o3_histchange_100hpa = float(meanvar(cnrm_o3t2_mean - cnrm_o3t1_mean, None, p=10000))

cnrm_o3_histchange_parcol_list = list()

for i in range(len(cnrm_o3)):
    
    cnrm_o3t2_pc=calc_parc_o3(cnrm_o3t2[i])
    cnrm_o3t1_pc=calc_parc_o3(cnrm_o3t1[i])
    
    cnrm_o3_histchange_parcol_list.append(float(latmean(cnrm_o3t2_pc) - latmean(cnrm_o3t1_pc)))

cnrm_o3_histchange_parcol = np.nanmean(cnrm_o3_histchange_parcol_list)

In [ ]:
cnrm_o3_hc = list()
for i in range(1,3):
    cnrm_o3_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/o3_Amon/CNRM-ESM2-1/r' + str(i + 1) + 
                                      'i1p1f2/gr/o3_Amon_CNRM-ESM2-1_hist-1950HC_r' + str(i + 1) + 'i1p1f2_gr_195001-201412.nc'))
    
cnrm_o3_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/o3_Amon/CNRM-ESM2-1/o3_AERmonZ_CNRM-ESM2-1_hist-1950HC_r1i1p1f2_grz_195001-201412.nc'))

In [ ]:
cnrm_o3_hct1 = list()
for i in range(len(cnrm_o3_hc)):
    cnrm_o3_hct1.append(prepdato3(cnrm_o3_hc[i], start=1958, end=1967))
    
cnrm_o3_hct2 = list()
for i in range(len(cnrm_o3_hc)):
    cnrm_o3_hct2.append(prepdato3(cnrm_o3_hc[i], start=1996, end=2005))
    
cnrm_o3_hct1_mean = sum(i for i in cnrm_o3_hct1) / len(cnrm_o3_hct1)
cnrm_o3_hct2_mean = sum(i for i in cnrm_o3_hct2) / len(cnrm_o3_hct2)

In [ ]:
cnrm_o3_hcchange_100hpa_list = list()
for i in range(len(cnrm_o3_hc)):
    cnrm_o3_hcchange_100hpa_list.append(float(meanvar(cnrm_o3_hct2[i] - cnrm_o3_hct1[i], None, p=10000)))
    
cnrm_o3_hcchange_100hpa = np.mean(cnrm_o3_hcchange_100hpa_list)

cnrm_o3_changeduetoHC_100hpa = [cnrm_o3_histchange_100hpa - cnrm_o3_hcchange_100hpa_list[i] for i in range(len(cnrm_o3_hcchange_100hpa_list))]

cnrm_o3_hcchange_parcol_list=list()

for i in range(len(cnrm_o3_hc)):
    
    cnrm_o3_hct2_pc=calc_parc_o3(cnrm_o3_hct2[i])
    cnrm_o3_hct1_pc=calc_parc_o3(cnrm_o3_hct1[i])
    
    
    cnrm_o3_hcchange_parcol_list.append(float(latmean(cnrm_o3_hct2_pc) - latmean(cnrm_o3_hct1_pc)))

cnrm_o3_hcchange_parcol = np.mean(cnrm_o3_hcchange_parcol_list)

cnrm_o3_changeduetoHC_100hpa = [cnrm_o3_histchange_100hpa - cnrm_o3_hcchange_100hpa_list[i] for i in range(len(cnrm_o3_hcchange_parcol_list))]
cnrm_o3_changeduetoHC_parcol = [cnrm_o3_histchange_parcol - cnrm_o3_hcchange_parcol_list[i] for i in range(len(cnrm_o3_hcchange_parcol_list))]

GFDL

In [ ]:
gfdl_o3 = list()

for i in range(1):
    gfdl_o3.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/o3_Amon/GFDL-ESM4/r' + str(i + 1) + 
                          'i1p1f1/gr1/o3_Amon_GFDL-ESM4_historical_r' + str(i + 1) + 'i1p1f1_gr1_195001-201412.nc'))
gfdl_o3.append(xr.open_dataset('/net/hydro/chemie/gchiodo/AerChemMIP/GFDL/o3_Amon_GFDL-ESM4_historical_r2i1p1f1_gr1_195001-201412.zm.nc'))
gfdl_o3.append(xr.open_dataset('/net/hydro/chemie/gchiodo/AerChemMIP/GFDL/o3_Amon_GFDL-ESM4_historical_r3i1p1f1_gr1_195001-201412.zm.nc'))

In [ ]:
gfdl_o3t1 = list()
for i in range(len(gfdl_o3)):
    gfdl_o3t1.append(prepdato3(gfdl_o3[i], start=1958, end=1967))
    
gfdl_o3t2 = list()
for i in range(len(gfdl_o3)):
    gfdl_o3t2.append(prepdato3(gfdl_o3[i], start=1996, end=2005))
    
gfdl_o3t1_mean = sum(i for i in gfdl_o3t1) / len(gfdl_o3t1)
gfdl_o3t2_mean = sum(i for i in gfdl_o3t2) / len(gfdl_o3t2)

In [ ]:
gfdl_o3_histchange_100hpa_list = list()
for i in range(len(gfdl_o3)):
    gfdl_o3_histchange_100hpa_list.append(float(meanvar(gfdl_o3t2[i] - gfdl_o3t1[i], None, p=10000)))

gfdl_o3_histchange_100hpa = float(meanvar(gfdl_o3t2_mean - gfdl_o3t1_mean, None, p=10000))


gfdl_o3_histchange_parcol_list = list()

for i in range(len(gfdl_o3)):
    
    gfdl_o3t2_pc=calc_parc_o3(gfdl_o3t2[i])
    gfdl_o3t1_pc=calc_parc_o3(gfdl_o3t1[i])
    
    gfdl_o3_histchange_parcol_list.append(float(latmean(gfdl_o3t2_pc) - latmean(gfdl_o3t1_pc)))
    
gfdl_o3_histchange_parcol = np.nanmean(gfdl_o3_histchange_parcol_list)

In [ ]:
gfdl_o3_hc = list()

for i in range(1):
    gfdl_o3_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/o3_Amon/GFDL-ESM4/r' + str(i + 1) + 
                                   'i1p1f1/gr1/o3_Amon_GFDL-ESM4_hist-1950HC_r' + str(i + 1) + 'i1p1f1_gr1_195001-201412.nc'))
gfdl_o3_hc.append(xr.open_dataset('/net/hydro/chemie/gchiodo/AerChemMIP/GFDL/hist-1950HC_D151/o3_Amon_GFDL-ESM4_hist-1950HC_D151.195001-201412.zm.nc'))
gfdl_o3_hc.append(xr.open_dataset('/net/hydro/chemie/gchiodo/AerChemMIP/GFDL/hist-1950HC_D201/o3_Amon_GFDL-ESM4_hist-1950HC_D201.195001-201412.zm.nc'))
gfdl_o3_hc[1] = gfdl_o3_hc[1].rename({"plev19": 'plev'})
gfdl_o3_hc[2] = gfdl_o3_hc[2].rename({"plev19": 'plev'})

In [ ]:
gfdl_o3_hct1 = list()
for i in range(len(gfdl_o3_hc)):
    gfdl_o3_hct1.append(prepdato3(gfdl_o3_hc[i], start=1958, end=1967))
    
gfdl_o3_hct2 = list()
for i in range(len(gfdl_o3_hc)):
    gfdl_o3_hct2.append(prepdato3(gfdl_o3_hc[i], start=1996, end=2005))
    
gfdl_o3_hct1_mean = sum(i for i in gfdl_o3_hct1) / len(gfdl_o3_hct1)
gfdl_o3_hct2_mean = sum(i for i in gfdl_o3_hct2) / len(gfdl_o3_hct2)

In [ ]:
gfdl_o3_hcchange_100hpa_list = list()
for i in range(len(gfdl_o3_hc)):
    gfdl_o3_hcchange_100hpa_list.append(float(meanvar(gfdl_o3_hct2[i] - gfdl_o3_hct1[i], None, p=10000)))
    
gfdl_o3_hcchange_100hpa = np.mean(gfdl_o3_hcchange_100hpa_list)

gfdl_o3_changeduetoHC_100hpa = [gfdl_o3_histchange_100hpa - gfdl_o3_hcchange_100hpa_list[i] for i in range(len(gfdl_o3_hcchange_100hpa_list))]


gfdl_o3_hcchange_parcol_list=list()

for i in range(len(gfdl_o3_hc)):
    
    gfdl_o3_hct2_pc=calc_parc_o3(gfdl_o3_hct2[i])
    gfdl_o3_hct1_pc=calc_parc_o3(gfdl_o3_hct1[i])
    
    
    gfdl_o3_hcchange_parcol_list.append(float(latmean(gfdl_o3_hct2_pc) - latmean(gfdl_o3_hct1_pc)))

gfdl_o3_hcchange_parcol = np.mean(gfdl_o3_hcchange_parcol_list)


gfdl_o3_changeduetoHC_parcol = [gfdl_o3_histchange_parcol - gfdl_o3_hcchange_parcol_list[i] for i in range(len(gfdl_o3_hcchange_parcol_list))]

GISS

For GISS there is ozone data missing for 2 of the fixed HC runs between 1964 and 1967

--> We take the "regular" time frame of 1955 - 1964 although this not perfectly consistent

This should not affect the result in a meaningful way, though

In [ ]:
giss_o3 = list()

# r7i1p3f1 is missing the temperature data from 1950 to 2001
# there is data for the ozone, but it is excluded to be able to compare it to the temperature trends seen!!!
# data obtained from interpolation using cdo ml2pl 
# for the first 6 runs
for i in range(6):
    giss_o3.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/o3/GISS-E2-1-G/r' +  str(i + 1) + 
                                   'i1p3f1/gn/o3_AERmon_GISS-E2-1-G_historical_r' + str(i + 1) + 'i1p3f1_gn_185001-201412_plev.nc'))
    
# for the runs 8 - 10
for i in range(7, 10):
    giss_o3.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/o3/GISS-E2-1-G/r' +  str(i + 1) + 
                                   'i1p3f1/gn/o3_AERmon_GISS-E2-1-G_historical_r' + str(i + 1) + 'i1p3f1_gn_185001-201412_plev.nc'))

In [ ]:
giss_o3t1 = list()
for i in range(len(giss_o3)):
    giss_o3t1.append(prepdato3(giss_o3[i], start=1955, end=1964))
    
giss_o3t2 = list()
for i in range(len(giss_o3)):
    giss_o3t2.append(prepdato3(giss_o3[i], start=1996, end=2005))
    
giss_o3t1_mean = sum(i for i in giss_o3t1) / len(giss_o3t1)
giss_o3t2_mean = sum(i for i in giss_o3t2) / len(giss_o3t2)

In [ ]:
giss_o3_histchange_100hpa_list = list()
for i in range(len(giss_o3)):
    giss_o3_histchange_100hpa_list.append(float(meanvar(giss_o3t2[i] - giss_o3t1[i], None, p=10000)))

giss_o3_histchange_100hpa = float(meanvar(giss_o3t2_mean - giss_o3t1_mean, None, p=10000))


giss_o3_histchange_parcol_list = list()

for i in range(len(giss_o3)):
    
    giss_o3t2_pc=calc_parc_o3(giss_o3t2[i])
    giss_o3t1_pc=calc_parc_o3(giss_o3t1[i])
    
    giss_o3_histchange_parcol_list.append(float(latmean(giss_o3t2_pc) - latmean(giss_o3t1_pc)))


giss_o3_histchange_parcol = np.nanmean(giss_o3_histchange_parcol_list)

In [ ]:
### Additonal data had some problems (cdo mergetime did not work) and workorund is/was needed.

giss_r2_t1 = xr.open_dataset("/net/ch4/chemie/wkonstantin/Additional_data/GISS-E2-1-G/hist-1950HC/o3/o3_AERmon_GISS-E2-1-G_hist-1950HC_r2i1p3f1_gn_195001-196412.nc")
giss_r2_t2 = xr.open_dataset("/net/ch4/chemie/wkonstantin/Additional_data/GISS-E2-1-G/hist-1950HC/o3/o3_AERmon_GISS-E2-1-G_hist-1950HC_r2i1p3f1_gn_199601-200512_plev_v2_nops.nc")



giss_r3_t1 = xr.open_dataset("/net/ch4/chemie/wkonstantin/Additional_data/GISS-E2-1-G/hist-1950HC/o3/o3_AERmon_GISS-E2-1-G_hist-1950HC_r3i1p3f1_gn_195001-196412.nc")
giss_r3_t2 = xr.open_dataset("/net/ch4/chemie/wkonstantin/Additional_data/GISS-E2-1-G/hist-1950HC/o3/o3_AERmon_GISS-E2-1-G_hist-1950HC_r3i1p3f1_gn_199601-200512.nc")

### r4 and r5 is treated normally (see below)

In [ ]:
### r1, r4, r5
giss_o3_hc = list()
for i in range(1):
    giss_o3_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/o3/GISS-E2-1-G/r' + str(i + 1) + 
                                      'i1p3f1/gn/o3_AERmon_GISS-E2-1-G_hist-1950HC_r' + str(i + 1) + 'i1p3f1_gn_195001-201412_plev.nc'))
    
giss_o3_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/Additional_data/GISS-E2-1-G/hist-1950HC/o3/o3_AERmon_GISS-E2-1-G_hist-1950HC_r5i1p3f1_gn_195001-201412.nc'))
giss_o3_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/Additional_data/GISS-E2-1-G/hist-1950HC/o3/o3_AERmon_GISS-E2-1-G_hist-1950HC_r4i1p3f1_gn_195001-201412_plev.nc'))

# in total 5 runs 

In [ ]:
def giss_special_o3(dataset, start, end):
    years = np.linspace(start, end, 10)
    years_str = [str(int(i)) for i in years]
    data_yearwise = list()
    for i in range(len(years_str)):
        data_yearwise.append(dataset["o3"].sel(time=years_str[i]).mean(dim="time").mean(dim="lon"))
    
    return sum(data_yearwise)/len(years_str)

In [ ]:
giss_o3_hct1 = list()
for i in range(len(giss_o3_hc)):
    giss_o3_hct1.append(prepdato3(giss_o3_hc[i], start=1955, end=1964))
    
giss_o3_hct1.append(giss_special_o3(giss_r2_t1, 1955, 1964))
giss_o3_hct1.append(giss_special_o3(giss_r3_t1, 1955, 1964))

giss_o3_hct2 = list()
for i in range(len(giss_o3_hc)):
    giss_o3_hct2.append(prepdato3(giss_o3_hc[i], start=1996, end=2005))
    
giss_o3_hct2.append(giss_special_o3(giss_r2_t2, 1996, 2005))
giss_o3_hct2.append(giss_special_o3(giss_r3_t2, 1996, 2005))
    
giss_o3_hct1_mean = sum(i for i in giss_o3_hct1) / len(giss_o3_hct1)
giss_o3_hct2_mean = sum(i for i in giss_o3_hct2) / len(giss_o3_hct2)

In [ ]:
giss_o3_hcchange_100hpa_list = list()
for i in range(len(giss_o3_hct1)):
    giss_o3_hcchange_100hpa_list.append(float(meanvar(giss_o3_hct2[i] - giss_o3_hct1[i], None, p=10000)))
    
giss_o3_hcchange_100hpa = np.mean(giss_o3_hcchange_100hpa_list)

giss_o3_changeduetoHC_100hpa = [giss_o3_histchange_100hpa - giss_o3_hcchange_100hpa_list[i] for i in range(len(giss_o3_hcchange_100hpa_list))]


giss_o3_hcchange_parcol_list=list()

for i in range(len(giss_o3_hc)):
    
    giss_o3_hct2_pc=calc_parc_o3(giss_o3_hct2[i])
    giss_o3_hct1_pc=calc_parc_o3(giss_o3_hct1[i])
    
    
    giss_o3_hcchange_parcol_list.append(float(latmean(giss_o3_hct2_pc) - latmean(giss_o3_hct1_pc)))

giss_o3_hcchange_parcol = np.mean(giss_o3_hcchange_parcol_list)


giss_o3_changeduetoHC_parcol = [giss_o3_histchange_parcol - giss_o3_hcchange_parcol_list[i] for i in range(len(giss_o3_hcchange_parcol_list))]

MRI

In [ ]:
mri_o3 = list()

for i in range(10):
    mri_o3.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/o3_Amon/MRI-ESM2-0/r' + str(i + 1) +
                         'i1p1f1/gn/o3_Amon_MRI-ESM2-0_historical_r' + str(i + 1) + 'i1p1f1_gn_195001-201412.nc'))

In [ ]:
mri_o3t1 = list()
for i in range(len(mri_o3)):
    mri_o3t1.append(prepdato3(mri_o3[i], start=1958, end=1967))
    
mri_o3t2 = list()
for i in range(len(mri_o3)):
    mri_o3t2.append(prepdato3(mri_o3[i], start=1996, end=2005))
    
mri_o3t1_mean = sum(i for i in mri_o3t1) / len(mri_o3t1)
mri_o3t2_mean = sum(i for i in mri_o3t2) / len(mri_o3t2)

In [ ]:
mri_o3_histchange_100hpa_list = list()
for i in range(len(mri_o3)):
    mri_o3_histchange_100hpa_list.append(float(meanvar(mri_o3t2[i] - mri_o3t1[i], None, p=10000)))

mri_o3_histchange_100hpa = float(meanvar(mri_o3t2_mean - mri_o3t1_mean, None, p=10000))


mri_o3_histchange_parcol_list = list()

for i in range(len(mri_o3)):
    
    mri_o3t2_pc=calc_parc_o3(mri_o3t2[i])
    mri_o3t1_pc=calc_parc_o3(mri_o3t1[i])
    
    mri_o3_histchange_parcol_list.append(float(latmean(mri_o3t2_pc) - latmean(mri_o3t1_pc)))

mri_o3_histchange_parcol = np.nanmean(mri_o3_histchange_parcol_list)

In [ ]:
mri_o3_hc = list()

for i in range(6):
    mri_o3_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/o3_Amon/MRI-ESM2-0/r' + str(i + 1) + 
                                     'i1p1f1/gn/o3_Amon_MRI-ESM2-0_hist-1950HC_r' + str(i + 1) + 'i1p1f1_gn_195001-201412.nc'))

In [ ]:
mri_o3_hct1 = list()
for i in range(len(mri_o3_hc)):
    mri_o3_hct1.append(prepdato3(mri_o3_hc[i], start=1958, end=1967))
    
mri_o3_hct2 = list()
for i in range(len(mri_o3_hc)):
    mri_o3_hct2.append(prepdato3(mri_o3_hc[i], start=1996, end=2005))
    
mri_o3_hct1_mean = sum(i for i in mri_o3_hct1) / len(mri_o3_hct1)
mri_o3_hct2_mean = sum(i for i in mri_o3_hct2) / len(mri_o3_hct2)

In [ ]:
mri_o3_hcchange_100hpa_list = list()
for i in range(len(mri_o3_hc)):
    mri_o3_hcchange_100hpa_list.append(float(meanvar(mri_o3_hct2[i] - mri_o3_hct1[i], None, p=10000)))
    
mri_o3_hcchange_100hpa = np.mean(mri_o3_hcchange_100hpa_list)

mri_o3_changeduetoHC_100hpa = [mri_o3_histchange_100hpa - mri_o3_hcchange_100hpa_list[i] for i in range(len(mri_o3_hcchange_100hpa_list))]


mri_o3_hcchange_parcol_list=list()

for i in range(len(mri_o3_hc)):
    
    mri_o3_hct2_pc=calc_parc_o3(mri_o3_hct2[i])
    mri_o3_hct1_pc=calc_parc_o3(mri_o3_hct1[i])
    
    
    mri_o3_hcchange_parcol_list.append(float(latmean(mri_o3_hct2_pc) - latmean(mri_o3_hct1_pc)))

mri_o3_hcchange_parcol = np.mean(mri_o3_hcchange_parcol_list)


mri_o3_changeduetoHC_parcol = [mri_o3_histchange_parcol - mri_o3_hcchange_parcol_list[i] for i in range(len(mri_o3_hcchange_parcol_list))]

UKESM

In [ ]:
ukesm_o3 = list()

for i in [1,2,3]:
    ukesm_o3.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/o3_Amon/UKESM1-0-LL/r' + str(i) +
                           'i1p1f2/gn/o3_Amon_UKESM1-0-LL_historical_r' + str(i) + 'i1p1f2_gn_195001-201412.nc'))
    
for i in [11,18,19]:
    ukesm_o3.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/o3_Amon/UKESM1-0-LL/r' + str(i) +
                           'i1p1f2/gn/o3_Amon_UKESM1-0-LL_historical_r' + str(i) + 'i1p1f2_gn_195001-201412.nc'))

In [ ]:
ukesm_o3t1 = list()
for i in range(len(ukesm_o3)):
    ukesm_o3t1.append(prepdato3(ukesm_o3[i], start=1958, end=1967))
    
ukesm_o3t2 = list()
for i in range(len(ukesm_o3)):
    ukesm_o3t2.append(prepdato3(ukesm_o3[i], start=1996, end=2005))
    
ukesm_o3t1_mean = sum(i for i in ukesm_o3t1) / len(ukesm_o3t1)
ukesm_o3t2_mean = sum(i for i in ukesm_o3t2) / len(ukesm_o3t2)

In [ ]:
ukesm_o3_histchange_100hpa_list = list()
for i in range(len(ukesm_o3)):
    ukesm_o3_histchange_100hpa_list.append(float(meanvar(ukesm_o3t2[i] - ukesm_o3t1[i], None, p=10000)))

ukesm_o3_histchange_100hpa = float(meanvar(ukesm_o3t2_mean - ukesm_o3t1_mean, None, p=10000))

ukesm_o3_histchange_parcol_list = list()

for i in range(len(ukesm_o3)):
    
    ukesm_o3t2_pc=calc_parc_o3(ukesm_o3t2[i])
    ukesm_o3t1_pc=calc_parc_o3(ukesm_o3t1[i])
    
    ukesm_o3_histchange_parcol_list.append(float(latmean(ukesm_o3t2_pc) - latmean(ukesm_o3t1_pc)))


ukesm_o3_histchange_parcol = np.nanmean(ukesm_o3_histchange_parcol_list)

In [ ]:
ukesm_o3_hc = list()

for i in range(3):
    ukesm_o3_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/o3_Amon/UKESM1-0-LL/r' + str(i + 1) +
                                       'i1p1f2/gn/o3_Amon_UKESM1-0-LL_hist-1950HC_r' + str(i + 1) + 'i1p1f2_gn_195001-201412.nc'))
    
ukesm_o3_hc.append(xr.open_dataset('/net/hydro/chemie/gchiodo/AerChemMIP/UKESM/hist-1950HC/o3_Amon_UKESM1-0-LL_hist-1950HC_r11i1p1f2_plev19_19500101-20150101.nc'))
ukesm_o3_hc.append(xr.open_dataset('/net/hydro/chemie/gchiodo/AerChemMIP/UKESM/hist-1950HC/o3_Amon_UKESM1-0-LL_hist-1950HC_r18i1p1f2_plev19_19500101-20150101.nc'))
ukesm_o3_hc.append(xr.open_dataset('/net/hydro/chemie/gchiodo/AerChemMIP/UKESM/hist-1950HC/o3_Amon_UKESM1-0-LL_hist-1950HC_r19i1p1f2_plev19_19500101-20150101.nc'))

In [ ]:
ukesm_o3_hct1 = list()
for i in range(len(ukesm_o3_hc)):
    ukesm_o3_hct1.append(prepdato3(ukesm_o3_hc[i], start=1958, end=1967))
    
ukesm_o3_hct2 = list()
for i in range(len(ukesm_o3_hc)):
    ukesm_o3_hct2.append(prepdato3(ukesm_o3_hc[i], start=1996, end=2005))
    
ukesm_o3_hct1_mean = sum(i for i in ukesm_o3_hct1) / len(ukesm_o3_hct1)
ukesm_o3_hct2_mean = sum(i for i in ukesm_o3_hct2) / len(ukesm_o3_hct2)

In [ ]:
ukesm_o3_hcchange_100hpa_list = list()
for i in range(len(ukesm_o3_hc)):
    ukesm_o3_hcchange_100hpa_list.append(float(meanvar(ukesm_o3_hct2[i] - ukesm_o3_hct1[i], None, p=10000)))
    
ukesm_o3_hcchange_100hpa = np.mean(ukesm_o3_hcchange_100hpa_list)

ukesm_o3_changeduetoHC_100hpa = [ukesm_o3_histchange_100hpa - ukesm_o3_hcchange_100hpa_list[i] for i in range(len(ukesm_o3_hcchange_100hpa_list))]


ukesm_o3_hcchange_parcol_list=list()

for i in range(len(ukesm_o3_hc)):
    
    ukesm_o3_hct2_pc=calc_parc_o3(ukesm_o3_hct2[i])
    ukesm_o3_hct1_pc=calc_parc_o3(ukesm_o3_hct1[i])
    
    ukesm_o3_hcchange_parcol_list.append(float(latmean(ukesm_o3_hct2_pc) - latmean(ukesm_o3_hct1_pc)))
    

ukesm_o3_hcchange_parcol = np.mean(ukesm_o3_hcchange_parcol_list)


ukesm_o3_changeduetoHC_parcol = [ukesm_o3_histchange_parcol - ukesm_o3_hcchange_parcol_list[i] for i in range(len(ukesm_o3_hcchange_parcol_list))]


## CMIP6 reference data set

In [ ]:
cmip6_o3 = xr.open_dataset("/net/reid/lhome/gchiodo/data/cmip6/vmro3_input4MIPs_ozone_CMIP_UReading-CCMI-1-0_gn_185001-201412.zm.nc",
                           decode_times=False)

In [ ]:
weights = np.cos(np.deg2rad(cmip6_o3["lat"]))
cmip6_o3_weighted = cmip6_o3.weighted(weights)
cmip6_o3_area_mean = cmip6_o3_weighted.mean(dim="lat")

In [ ]:
cmip6_o3_100hpa = cmip6_o3_area_mean["vmro3"].sel(plev=100)

cmip6_o3_100hpa_yr_means = list()
for i in range(0, len(cmip6_o3_100hpa), 12):
    cmip6_o3_100hpa_yr_means.append(float((cmip6_o3_100hpa[i] + cmip6_o3_100hpa[i + 1] + cmip6_o3_100hpa[i + 2] + 
                              cmip6_o3_100hpa[i + 3] + cmip6_o3_100hpa[i + 4] + cmip6_o3_100hpa[i + 5] + 
                              cmip6_o3_100hpa[i + 6] + cmip6_o3_100hpa[i + 7] + cmip6_o3_100hpa[i + 8] + 
                              cmip6_o3_100hpa[i + 9] + cmip6_o3_100hpa[i + 10] + cmip6_o3_100hpa[i + 11]) / 12) )

In [ ]:
cmip6_o3_parcol=calc_parc_o3(cmip6_o3_area_mean["vmro3"])

cmip6_o3_parcol_yr_means = list()
for i in range(0, len(cmip6_o3_parcol), 12):
    cmip6_o3_parcol_yr_means.append(float((cmip6_o3_parcol[i] + cmip6_o3_parcol[i + 1] + cmip6_o3_parcol[i + 2] + 
                              cmip6_o3_parcol[i + 3] + cmip6_o3_parcol[i + 4] + cmip6_o3_parcol[i + 5] + 
                              cmip6_o3_parcol[i + 6] + cmip6_o3_parcol[i + 7] + cmip6_o3_parcol[i + 8] + 
                              cmip6_o3_parcol[i + 9] + cmip6_o3_parcol[i + 10] + cmip6_o3_parcol[i + 11]) / 12) )

## JRA55

In [ ]:
jra55_o3_mass_mix = xr.open_dataset('/net/hydro/chemie/mfriedel/Data/jra55_ozone_195801_201212.nc')["OZONE_GDS0_ISBL_S113"]

In [ ]:
#conversion from mass mixing ratio to mol mixing ratio
Mair = 28.9647
Mo3 = 47.997

jra55_o3 = jra55_o3_mass_mix * Mair / Mo3

In [ ]:
jra55_o3_zonal_mean = jra55_o3.mean(dim="g0_lon_3")

In [ ]:
jra55_weights = np.cos(np.deg2rad(jra55_o3_zonal_mean["g0_lat_2"]))
jra55_weighted = jra55_o3_zonal_mean.weighted(jra55_weights)
jra55_o3_global_mean = jra55_weighted.mean(dim="g0_lat_2")

In [ ]:
jra55_o3_global_mean_parc=calc_parc_o3(jra55_o3_global_mean*1e-6)

#jra55_o3_global_mean_yearly = jra55_o3_global_mean.groupby('initial_time0_hours.year').mean()

jra55_o3_global_mean_yearly = list()
years = 2012 - 1957
for i in range(years):
    jra55_o3_global_mean_yearly.append((jra55_o3_global_mean[i*12] + jra55_o3_global_mean[i*12 + 1] + jra55_o3_global_mean[i*12 + 2] +
    jra55_o3_global_mean[i*12 + 3] + jra55_o3_global_mean[i*12 + 4] + jra55_o3_global_mean[i*12 + 5] + 
    jra55_o3_global_mean[i*12 + 6] + jra55_o3_global_mean[i*12 + 7] + jra55_o3_global_mean[i*12 + 8] +
    jra55_o3_global_mean[i*12 + 9] + jra55_o3_global_mean[i*12 + 10] + jra55_o3_global_mean[i*12 + 11])/12)



jra55_o3_global_mean_yearly_parc = jra55_o3_global_mean_parc.groupby('initial_time0_hours.year').mean()

## Surface temperature coupled runs

In [ ]:
cesm_tas = list()

### First three ensemble members start in 1850, data points at the middle of the month ###
for i in range(3):
    cesm_tas.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/tas/CESM2-WACCM/r' + str(i + 1)
                     + 'i1p1f1/gn/tas_Amon_CESM2-WACCM_historical_r' + str(i + 1) + 'i1p1f1_gn_185001-201412.nc'))
    
### Those 12 ensemble members start only in 1950, data points for a month at the beginning of the next month!!! ###
for i in range(3):
    for j in range(4):
        cesm_tas.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/Additional_data/CESM2-WACCM/wa6_ic' + str(i + 1) + '.00' + str(j + 1) + 
                              '/wa6_ic' + str(i + 1) + '.00' + str(j + 1) + '.cam.h0.1950-2014_T2Ms_TREFHT.nc'))

In [ ]:
cesm_tas_hc = list()
### data points at the middle of the month ###
cesm_tas_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/tas/CESM2-WACCM//r1i1p1f1/gn/tas_Amon_CESM2-WACCM_hist-1950HC_r1i1p1f1_gn_195001-201412.nc'))

### data points for a month at the beginning of the next month!!! ###
cesm_tas_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/Additional_data/CESM2-WACCM/wa6_ens.HC1950.002/wa6_ens.HC1950.002.cam.h0.1950-2014_T2Ms_TREFHT.nc'))
cesm_tas_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/Additional_data/CESM2-WACCM/wa6_ens.HC1950.003/wa6_ens.HC1950.003.cam.h0.1950-2014_T2Ms_TREFHT.nc'))
cesm_tas_hc.append(xr.open_dataset('/net/reid/lhome/gchiodo/data/Columbia/wa6_ens.HC1950.004/wa6_ens.HC1950.004.cam.h0.1950-2014_T2Ms_TREFHT.nc'))

In [ ]:
cnrm_tas = list()
for i in range(11):
    cnrm_tas.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/tas/CNRM-ESM2-1/r' + str(i + 1)
                        + 'i1p1f2/gr/tas_Amon_CNRM-ESM2-1_historical_r' + str(i + 1) + 'i1p1f2_gr_185001-201412.nc'))

In [ ]:
cnrm_tas_hc = list()
cnrm_tas_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/tas/CNRM-ESM2-1/r1i1p1f2/gr/tas_Amon_CNRM-ESM2-1_hist-1950HC_r1i1p1f2_gr_195001-201412.nc'))
cnrm_tas_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/tas/CNRM-ESM2-1/r2i1p1f2/gr/tas_Amon_CNRM-ESM2-1_hist-1950HC_r2i1p1f2_gr_195001-201412.nc'))
cnrm_tas_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/tas/CNRM-ESM2-1/r3i1p1f2/gr/tas_Amon_CNRM-ESM2-1_hist-1950HC_r3i1p1f2_gr_195001-201412.nc'))



In [ ]:
gfdl_tas = list()
for i in range(3):
    gfdl_tas.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/tas/GFDL-ESM4/r' + str(i + 1) + 
                          'i1p1f1/gr1/tas_Amon_GFDL-ESM4_historical_r' + str(i + 1) + 'i1p1f1_gr1_195001-201412.nc'))

In [ ]:
gfdl_tas_hc = list()
gfdl_tas_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/tas/GFDL-ESM4/r1i1p1f1/gr1/tas_Amon_GFDL-ESM4_hist-1950HC_r1i1p1f1_gr1_195001-201412.nc'))
gfdl_tas_hc.append(xr.open_dataset('/net/hydro/chemie/gchiodo/AerChemMIP/GFDL/hist-1950HC_D151/tas_Amon_GFDL-ESM4_hist-1950HC_D151.195001-201412.nc'))
gfdl_tas_hc.append(xr.open_dataset('/net/hydro/chemie/gchiodo/AerChemMIP/GFDL/hist-1950HC_D201/tas_Amon_GFDL-ESM4_hist-1950HC_D201.195001-201412.nc'))

In [ ]:
giss_tas = list()

# r7i1p3f1 is missing the data from 1950 to 2001
# for the first 6 runs
for i in range(6):
    giss_tas.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/tas/GISS-E2-1-G/r' + str(i + 1) + 
                          'i1p3f1/gn/tas_Amon_GISS-E2-1-G_historical_r' + str(i + 1) + 'i1p3f1_gn_190101-201412.nc'))
    
# for the runs 8 - 10
for i in range(7, 10):
    giss_tas.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/tas/GISS-E2-1-G/r' + str(i + 1) + 
                          'i1p3f1/gn/tas_Amon_GISS-E2-1-G_historical_r' + str(i + 1) + 'i1p3f1_gn_190101-201412.nc'))

In [ ]:
giss_tas_hc = list()
giss_tas_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/tas/GISS-E2-1-G/r1i1p3f1/gn/tas_Amon_GISS-E2-1-G_hist-1950HC_r1i1p3f1_gn_195001-201412.nc'))

### Additional ensemble members ###
# have the same interpolation as all other GISS ensemble members #
for i in range(4):
    giss_tas_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/Additional_data/GISS-E2-1-G/hist-1950HC/tas/tas_Amon_GISS-E2-1-G_hist-1950HC_r' +
                                   str(i + 2) + 'i1p3f1_gn_195001-201412.nc'))

In [ ]:
mri_tas = list()

for i in range(10):
    mri_tas.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/tas/MRI-ESM2-0/r' + str(i + 1) +
                         'i1p1f1/gn/tas_Amon_MRI-ESM2-0_historical_r' + str(i + 1) + 'i1p1f1_gn_185001-201412.nc'))

In [ ]:
mri_tas_hc = list()
for i in range(6):
    mri_tas_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/tas/MRI-ESM2-0/r' + str(i + 1) + 
                            'i1p1f1/gn/tas_Amon_MRI-ESM2-0_hist-1950HC_r' +  str(i + 1) + 'i1p1f1_gn_195001-201412.nc'))

In [ ]:
ukesm_tas = list()

for i in [1,2,3]:
    ukesm_tas.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/tas/UKESM1-0-LL/r' + str(i) +
                           'i1p1f2/gn/tas_Amon_UKESM1-0-LL_historical_r' + str(i) + 'i1p1f2_gn_195001-201412.nc'))
    
for i in [11, 18, 19]:
    ukesm_tas.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/historical/tas/UKESM1-0-LL/r' + str(i) +
                           'i1p1f2/gn/tas_Amon_UKESM1-0-LL_historical_r' + str(i) + 'i1p1f2_gn_195001-201412.nc'))


In [ ]:
ukesm_tas_hc = list()

for i in range(3):
    ukesm_tas_hc.append(xr.open_dataset('/net/ch4/chemie/wkonstantin/cimp6_data/hist-1950HC/tas/UKESM1-0-LL/r' + str(i + 1) +
                              'i1p1f2/gn/tas_Amon_UKESM1-0-LL_hist-1950HC_r' + str(i + 1) + 'i1p1f2_gn_195001-201412.nc'))
    
ukesm_tas_hc.append(xr.open_dataset('/net/hydro/chemie/gchiodo/AerChemMIP/UKESM/hist-1950HC/tas_Amon_UKESM1-0-LL_hist-1950HC_r11i1p1f2_19500101-20150101.nc'))
ukesm_tas_hc.append(xr.open_dataset('/net/hydro/chemie/gchiodo/AerChemMIP/UKESM/hist-1950HC/tas_Amon_UKESM1-0-LL_hist-1950HC_r18i1p1f2_19500101-20150101.nc'))
ukesm_tas_hc.append(xr.open_dataset('/net/hydro/chemie/gchiodo/AerChemMIP/UKESM/hist-1950HC/tas_Amon_UKESM1-0-LL_hist-1950HC_r19i1p1f2_19500101-20150101.nc'))

## Data preparation

In [ ]:
### CESM needs to be treated differently, since not all data is in the same format
### Operations are done by indexing and with the built-in function "sel" (which is used for all the other models)
cesm_global_yr_mean = list()
for i in range(len(cesm_tas)):
    cesm_global_yr_mean.append(meanvar_cesm(cesm_tas[i], var="tas", p=None, lat_low=-90, lat_high=90))
cesm_global_indiv_warming = list()
for i in range(len(cesm_global_yr_mean)):
    if i in (0,1,2):
        t1 = cesm_global_yr_mean[i][(1955-1850):(1964-1850+1)].mean()
        t2 = cesm_global_yr_mean[i][(1996-1850):(2005-1850+1)].mean()
        cesm_global_indiv_warming.append(t2-t1)
    else:
        t1 = cesm_global_yr_mean[i][(1955-1950):(1964-1950+1)].mean()
        t2 = cesm_global_yr_mean[i][(1996-1950):(2005-1950+1)].mean()
        cesm_global_indiv_warming.append(t2-t1)

In [ ]:
cesm_hc_global_yr_mean = list()
for i in range(len(cesm_tas_hc)):
    cesm_hc_global_yr_mean.append(meanvar_cesm(cesm_tas_hc[i], var="tas", p=None, lat_low=-90, lat_high=90))
cesm_hc_global_indiv_warming = list()
for i in range(len(cesm_hc_global_yr_mean)):
    t1 = cesm_hc_global_yr_mean[i][(1955-1950):(1964-1950+1)].mean()
    t2 = cesm_hc_global_yr_mean[i][(1996-1950):(2005-1950+1)].mean()
    cesm_hc_global_indiv_warming.append(t2-t1)

In [ ]:
cnrm_global_indiv_warming = warming_indiv(cnrm_tas)

In [ ]:
cnrm_hc_global_indiv_warming = warming_indiv(cnrm_tas_hc)

In [ ]:
cnrm_global_all_warming,cnrm_global_all_warming_err = warming_all(cnrm_tas)

In [ ]:
cnrm_hc_global_all_warming, cnrm_hc_global_all_warming_err = warming_all(cnrm_tas_hc)

In [ ]:
gfdl_global_indiv_warming = warming_indiv(gfdl_tas)

In [ ]:
gfdl_hc_global_indiv_warming = warming_indiv(gfdl_tas_hc)

In [ ]:
gfdl_global_all_warming,gfdl_global_all_warming_err = warming_all(gfdl_tas)

In [ ]:
gfdl_hc_global_all_warming, gfdl_hc_global_all_warming_err = warming_all(gfdl_tas_hc)

In [ ]:
giss_global_indiv_warming = warming_indiv(giss_tas)

In [ ]:
giss_hc_global_indiv_warming = warming_indiv(giss_tas_hc)

In [ ]:
mri_global_indiv_warming = warming_indiv(mri_tas)

In [ ]:
mri_hc_global_indiv_warming = warming_indiv(mri_tas_hc)

In [ ]:
ukesm_global_indiv_warming = warming_indiv(ukesm_tas)

In [ ]:
ukesm_hc_global_indiv_warming = warming_indiv(ukesm_tas_hc)

# ERF Yearly means
## Data preparation - Global

In [ ]:
rlut_control_yr_mean = list()
for i in range(len(rlut_control)):
    rlut_control_yr_mean.append(meanvar(rlut_control[i], var="rlut"))

In [ ]:
rsdt_control_yr_mean = list()
for i in range(len(rsdt_control)):
    rsdt_control_yr_mean.append(meanvar(rsdt_control[i], var="rsdt"))

In [ ]:
rsut_control_yr_mean = list()
for i in range(len(rsut_control)):
    rsut_control_yr_mean.append(meanvar(rsut_control[i], var="rsut"))

In [ ]:
rlut_hc_yr_mean = list()
for i in range(len(rlut_hc)):
    rlut_hc_yr_mean.append(meanvar(rlut_hc[i], var="rlut"))

In [ ]:
rsdt_hc_yr_mean = list()
for i in range(len(rsdt_hc)):
    rsdt_hc_yr_mean.append(meanvar(rsdt_hc[i], var="rsdt"))

In [ ]:
rsut_hc_yr_mean = list()
for i in range(len(rsut_hc)):
    rsut_hc_yr_mean.append(meanvar(rsut_hc[i], var="rsut"))

In [ ]:
# Net TOA radiation for control experiments, positive in, negative out
control_net_rad = list()
for i in range(len(rlut_control_yr_mean)):
    control_net_rad.append((rsdt_control_yr_mean[i] - rsut_control_yr_mean[i]) - rlut_control_yr_mean[i])

In [ ]:
# Net TOA radiation for hc experiments, positive in, negative out
hc_net_rad = list()
for i in range(len(rlut_hc_yr_mean)):
    hc_net_rad.append((rsdt_hc_yr_mean[i] - rsut_hc_yr_mean[i]) - rlut_hc_yr_mean[i])

In [ ]:
# Net radiative forcing for HCs
rf_hc = list()
for i in range(len(rlut_control_yr_mean)):
    rf_hc.append(hc_net_rad[i] - control_net_rad[i])

## Global mean ERF

In [ ]:
# Mean RF over the whole period

print("Mean RF of HCs in piClim experiments over the whole period")
for i in range(6):
    print(names[i] + ": " + str(np.around(float(np.mean(rf_hc[i])), 3)) + " W m-2")

In [ ]:
mean_global_erf_hc_all = list()
stdev_global_erf_hc_all = list()
for i in range(6):
    mean_global_erf_hc_all.append(float(np.mean(rf_hc[i])))
    stdev_global_erf_hc_all.append(float(np.std(rf_hc[i])/np.sqrt(len(rf_hc[i]))))

In [ ]:
mean_global_erf_hc_wo10yrs = list()
stdev_global_erf_hc_wo10yrs = list()
for i in range(6):
    mean_global_erf_hc_wo10yrs.append(float(np.mean(rf_hc[i][10:])))
    stdev_global_erf_hc_wo10yrs.append(float(np.std(rf_hc[i][10:])/np.sqrt(len(rf_hc[i][10:]))))

In [ ]:
with open('piclim_erf_hc_global_all.pkl', 'wb') as f:
    pickle.dump([mean_global_erf_hc_all, stdev_global_erf_hc_all], f)
    
with open('piclim_erf_hc_global_wo10yrs.pkl', 'wb') as f:
    pickle.dump([mean_global_erf_hc_wo10yrs, stdev_global_erf_hc_wo10yrs], f)

In [ ]:
# Mean ERF over the whole period

print("Mean RF of HCs in piClim experiments over the last 30 years")
for i in range(6):
    print(names[i] + ": " + str(np.around(float(np.mean(rf_hc[i][-30:])), 3)) + " W m-2")

# piClim ERF vs coupled runs ozone depletion

## O3 change due to HCs

In [ ]:
markers = ["o", "^", "s", "*", "P", "X", "p"]
colors = ["royalblue", "peru", "forestgreen", "firebrick", "purple", "sienna"]

xdata = np.array([cesm_o3_histchange_100hpa,
                  cnrm_o3_histchange_100hpa,
                  gfdl_o3_histchange_100hpa,
                  giss_o3_histchange_100hpa,
                  mri_o3_histchange_100hpa,
                  ukesm_o3_histchange_100hpa]) * 10**6

xerror = np.array([np.std(cesm_o3_histchange_100hpa_list),
                   np.std(cnrm_o3_histchange_100hpa_list),
                   np.std(gfdl_o3_histchange_100hpa_list),
                   np.std(giss_o3_histchange_100hpa_list),
                   np.std(mri_o3_histchange_100hpa_list),
                   np.std(ukesm_o3_histchange_100hpa_list)]) * 10**6

ydata = mean_global_erf_hc_all

yerror = stdev_global_erf_hc_all

cmip6_ref = (np.mean(cmip6_o3_100hpa_yr_means[(1996 - 1850):(2005 - 1850)]) -
             np.mean(cmip6_o3_100hpa_yr_means[(1958 - 1850):(1967 - 1850)])) * 10**6

jra55_ref = (np.mean([i.sel(lv_ISBL1 = 100) for i in jra55_o3_global_mean_yearly[(1996 - 1958):(2005 - 1958)]]) -
             np.mean([i.sel(lv_ISBL1 = 100) for i in jra55_o3_global_mean_yearly[(1958 - 1958):(1967 - 1958)]]))
                 
fig = plt.figure(figsize=(9, 6), dpi=300)
ax = plt.gca()
for i in range(len(xdata)):
    plt.errorbar(xdata[i], ydata[i], xerr=xerror[i], yerr=yerror[i], marker=markers[i],
                 markersize=8, zorder=2, capsize = 3, capthick=1, elinewidth=1, color=colors[i])
    
ax.set_xlim(right=0.01)

plt.legend(names,fontsize=12)

ymin, ymax = plt.ylim()
plt.plot([cmip6_ref, cmip6_ref], [ymin, ymax], c="darkred", linestyle='--')
plt.text(cmip6_ref - 0.03, ymin + 0.02, 'CMIP6 ozone\nforcing data set', c="darkred", fontsize=12, zorder=1)

plt.plot([jra55_ref, jra55_ref], [ymin, ymax], c="indianred", linestyle='--')
plt.text(jra55_ref + 0.003, ymin + 0.02, 'JRA-55', c="indianred", fontsize=12, zorder=1)

x = np.linspace(min(xdata), 0.01, 200)
slope, intercept, r, p, se = stats.linregress(xdata, ydata)
#reg = LinearRegression(fit_intercept=True).fit([[j] for j in xdata], [[j] for j in ydata])
plt.plot(x, x*slope +intercept, color='k', linestyle=':', label='')


ax.text(-0.172,0.4, 'R = ' + str("%.3f" % r) + '\n s = ' + str("%.3f" % slope)+ '\n p = ' + str("%.3f" % p)  , horizontalalignment='left',fontsize=12,color='black')


plt.plot([0, 0], [0, reg.intercept_[0]], color="k", linewidth=0.6, zorder=1)
plt.plot(0, reg.intercept_[0], color="k", marker="o", markersize=4)
plt.plot([-0.18, 0], [reg.intercept_[0],reg.intercept_[0]], color="k", linewidth=0.6 )



plt.text(-0.005, 0.4, str(np.round(reg.intercept_[0], 3)))


plt.axhline(y=0, color='grey', linestyle='-')    
    
plt.xlabel(r"Historical ozone mixing ratio change at 100 hPa in coupled simulations /ppm", fontsize=12)
plt.ylabel(r"piClim ERF of HCs /W m$^{-2}$", fontsize=12)    

plt.savefig("hc_erf_piclim_vs_coupledsim_ozone_depletion_historical_100hPa.png", bbox_inches="tight")

plt.savefig("hc_erf_piclim_vs_coupledsim_ozone_depletion_historical_100hPa.svg", bbox_inches="tight", facecolor="white")

In [ ]:
import scipy

markers = ["o", "^", "s", "*", "P", "X", "p"]
colors = ["royalblue", "peru", "forestgreen", "firebrick", "purple", "sienna"]

xdata = np.array([cesm_o3_histchange_parcol,
                  cnrm_o3_histchange_parcol,
                  gfdl_o3_histchange_parcol,
                  giss_o3_histchange_parcol,
                  mri_o3_histchange_parcol,
                  ukesm_o3_histchange_parcol]) 

xerror = np.array([np.std(cesm_o3_histchange_parcol_list),
                   np.std(cnrm_o3_histchange_parcol_list),
                   np.std(gfdl_o3_histchange_parcol_list),
                   np.std(giss_o3_histchange_100hpa_list),
                   np.std(mri_o3_histchange_parcol_list),
                   np.std(ukesm_o3_histchange_parcol_list)]) 

ydata = mean_global_erf_hc_all

yerror = stdev_global_erf_hc_all

cmip6_ref = (np.mean(cmip6_o3_parcol_yr_means[(1996 - 1850):(2005 - 1850)]) -
             np.mean(cmip6_o3_parcol_yr_means[(1958 - 1850):(1967 - 1850)])) 


jra55_ref = (np.mean(jra55_o3_global_mean_yearly_parc[(1996 - 1958):(2005 - 1958)]) -
             np.mean(jra55_o3_global_mean_yearly_parc[(1958 - 1958):(1967 - 1958)]))


                 
fig = plt.figure(figsize=(9, 6), dpi=300)
ax = plt.gca()
for i in range(len(xdata)):
    plt.errorbar(xdata[i], ydata[i], xerr=xerror[i], yerr=yerror[i], marker=markers[i],color=colors[i],
                 markersize=8, zorder=2, capsize = 3, capthick=1, elinewidth=1) 
    
plt.legend(names,fontsize=12)

ymin, ymax = plt.ylim()
plt.plot([cmip6_ref, cmip6_ref], [ymin, ymax], c="darkred", linestyle='--')
plt.text(cmip6_ref + 0.2, ymin + 0.02, 'CMIP6 ozone\nforcing data set', c="darkred", fontsize=12, zorder=1)

plt.plot([jra55_ref, jra55_ref], [ymin, ymax], c="indianred", linestyle='--')
plt.text(jra55_ref + 0.2, ymin + 0.02, 'JRA-55', c="indianred", fontsize=12, zorder=1)

x = np.linspace(min(xdata), 0.01, 200)
slope, intercept, r, p, se = stats.linregress(xdata, ydata)
reg = LinearRegression(fit_intercept=True).fit([[j] for j in xdata], [[j] for j in ydata])
plt.plot(x, x*slope +intercept, color='k', linestyle=':', label='')


ax.text(-17.5,0.3, 'R = ' + str("%.3f" % r) + '\n s = ' + str("%.3f" % slope)+ '\n p = ' + str("%.3f" % p)  , horizontalalignment='left',fontsize=12,color='black')


plt.plot([0, 0], [0, reg.intercept_[0]], color="k", linewidth=0.6, zorder=1)
plt.plot(0, reg.intercept_[0], color="k", marker="o", markersize=4)
plt.plot([-19, 0], [reg.intercept_[0],reg.intercept_[0]], color="k", linewidth=0.6 )



plt.text(-0.005, 0.42, str(np.round(reg.intercept_[0], 2)))

plt.xlim(-18,1)

plt.axhline(y=0, color='grey', linestyle='-')    
    
plt.xlabel(r"Historical partial ozone column change (30-100 hPa) in coupled simulations /DU", fontsize=12)
plt.ylabel(r"piClim ERF of HCs /W m$^{-2}$", fontsize=12)    

plt.savefig("hc_erf_piclim_vs_coupledsim_ozone_depletion_historical_parcol.30-100hPa.png", bbox_inches="tight")

# piClim ERF vs coupled runs tas change

## tas change due to HCs

In [ ]:
from scipy import stats
import scipy
import string




markers = ["o", "^", "s", "*", "P", "X", "p"]
colors = ["royalblue", "peru", "forestgreen", "firebrick", "purple", "sienna"]

xdata = [np.mean(cesm_global_indiv_warming) - np.mean(cesm_hc_global_indiv_warming),
         np.mean(cnrm_global_indiv_warming) - np.mean(cnrm_hc_global_indiv_warming),
         np.mean(gfdl_global_indiv_warming) - np.mean(gfdl_hc_global_indiv_warming),
         np.mean(giss_global_indiv_warming) - np.mean(giss_hc_global_indiv_warming),
         np.mean(mri_global_indiv_warming) - np.mean(mri_hc_global_indiv_warming),
         np.nanmean(ukesm_global_indiv_warming) - np.nanmean(ukesm_hc_global_indiv_warming)]

xerror = np.sqrt(np.array([np.var(cesm_global_indiv_warming) + np.var(cesm_hc_global_indiv_warming),
         np.var(cnrm_global_indiv_warming) + np.var(cnrm_hc_global_indiv_warming),
         np.var(gfdl_global_indiv_warming) + np.var(gfdl_hc_global_indiv_warming),
         np.var(giss_global_indiv_warming) + np.var(giss_hc_global_indiv_warming),
         np.var(mri_global_indiv_warming) + np.var(mri_hc_global_indiv_warming),
         np.nanvar(ukesm_global_indiv_warming) + np.nanvar(ukesm_hc_global_indiv_warming)]))



xdata_ind=[]
ydata_ind=[]
colors_ind=[]

i=0
 
for j in range(len(cesm_hc_global_indiv_warming)):
    xdata_ind.append(np.mean(cesm_global_indiv_warming) - cesm_hc_global_indiv_warming[j])
    ydata_ind.append(mean_global_erf_hc_all[0])
    colors_ind.append(colors[0])
    if i==0: start_cesm=i
    i=i+1
    
for j in range(len(cnrm_hc_global_indiv_warming)):
    xdata_ind.append(np.mean(cnrm_global_indiv_warming) - cnrm_hc_global_indiv_warming[j])
    ydata_ind.append(mean_global_erf_hc_all[1])
    colors_ind.append(colors[1])
    if j==0: start_cnrm=i
    i=i+1
    
for j in range(len(gfdl_hc_global_indiv_warming)):
    xdata_ind.append(np.mean(gfdl_global_indiv_warming) - gfdl_hc_global_indiv_warming[j])
    ydata_ind.append(mean_global_erf_hc_all[2])
    colors_ind.append(colors[2])
    if j==0: start_gfdl=i
    i=i+1
    
for j in range(len(giss_hc_global_indiv_warming)):
    xdata_ind.append(np.mean(giss_global_indiv_warming) - giss_hc_global_indiv_warming[j])
    ydata_ind.append(mean_global_erf_hc_all[3])
    colors_ind.append(colors[3])
    if j==0: start_giss=i
    i=i+1
    
for j in range(len(mri_hc_global_indiv_warming)):
    xdata_ind.append(np.mean(mri_global_indiv_warming) - mri_hc_global_indiv_warming[j])
    ydata_ind.append(mean_global_erf_hc_all[4])
    colors_ind.append(colors[4])
    if j==0: start_mri=i
    i=i+1
    
for j in range(len(ukesm_hc_global_indiv_warming)):
    xdata_ind.append(np.mean(ukesm_global_indiv_warming) - ukesm_hc_global_indiv_warming[j])
    ydata_ind.append(mean_global_erf_hc_all[5])
    colors_ind.append(colors[5])
    if j==0: start_ukesm=i
    i=i+1
   
xdata_err_ind=[]
xdata_err_ind.append([np.min(xdata_ind[0:len(cesm_hc_global_indiv_warming)]-np.mean(xdata_ind[0:len(cesm_hc_global_indiv_warming)])),np.max(xdata_ind[0:len(cesm_hc_global_indiv_warming)]-np.mean(xdata_ind[0:len(cesm_hc_global_indiv_warming)]))] )
xdata_err_ind.append([np.min(xdata_ind[start_cnrm:start_gfdl]-np.mean(xdata_ind[start_cnrm:start_gfdl])),np.max(xdata_ind[start_cnrm:start_gfdl]-np.mean(xdata_ind[start_cnrm:start_gfdl]))] )
xdata_err_ind.append([np.min(xdata_ind[start_gfdl:start_giss]-np.mean(xdata_ind[start_gfdl:start_giss])),np.max(xdata_ind[start_gfdl:start_giss]-np.mean(xdata_ind[start_gfdl:start_giss]))] )
xdata_err_ind.append([np.min(xdata_ind[start_giss:start_mri]-np.mean(xdata_ind[start_giss:start_mri])),np.max(xdata_ind[start_giss:start_mri]-np.mean(xdata_ind[start_giss:start_mri]))] )
xdata_err_ind.append([np.min(xdata_ind[start_mri:start_ukesm]-np.mean(xdata_ind[start_mri:start_ukesm])),np.max(xdata_ind[start_mri:start_ukesm]-np.mean(xdata_ind[start_mri:start_ukesm]))] )
xdata_err_ind.append([np.min(xdata_ind[start_ukesm:len(xdata_ind)]-np.mean(xdata_ind[start_ukesm:len(xdata_ind)])),np.max(xdata_ind[start_ukesm:len(xdata_ind)]-np.mean(xdata_ind[start_ukesm:len(xdata_ind)]))] )



t_cesm, p_cesm = scipy.stats.ttest_ind(cesm_global_indiv_warming,cesm_hc_global_indiv_warming,  equal_var=False)
t_cnrm, p_cnrm = scipy.stats.ttest_ind(cnrm_global_all_warming,cnrm_hc_global_all_warming,  equal_var=False)
t_gfdl, p_gfdl= scipy.stats.ttest_ind(gfdl_global_indiv_warming,gfdl_hc_global_indiv_warming,  equal_var=False)
t_giss, p_giss = scipy.stats.ttest_ind(giss_global_indiv_warming,giss_hc_global_indiv_warming,  equal_var=False)
t_mri, p_mri = scipy.stats.ttest_ind(mri_global_indiv_warming,mri_hc_global_indiv_warming,  equal_var=False)
t_ukesm, p_ukesm = scipy.stats.ttest_ind(ukesm_global_indiv_warming, ukesm_hc_global_indiv_warming,  equal_var=False)

#print(p_cesm)
#print(p_cnrm)
#print(p_gfdl)
#print(p_giss)
#print(p_mri)
#print(p_ukesm)



ydata = mean_global_erf_hc_all

yerror = stdev_global_erf_hc_all
                 
fig, ax = plt.subplots(1,2, figsize=(8,5), constrained_layout=True, sharex='col', sharey='row', gridspec_kw={'width_ratios': [8,1]})

#ax[0] = plt.gca()
for i in range(len(xdata)):
    ax[0].errorbar(ydata[i], xdata[i], xerr=yerror[i], yerr=np.reshape(np.abs(xdata_err_ind[i]),(2,1)), marker=markers[i],
                 markersize=8, zorder=2, capsize = 3, capthick=1, elinewidth=1, color=colors[i], alpha=0.7)
    
    
ax[0].legend(names, fontsize=15)
    
for i in range(len(xdata_ind)):    
    ax[0].plot(ydata_ind[i], xdata_ind[i], linestyle='', marker='o', color=colors_ind[i], markersize=4, label='')
    
    
slope, intercept, r, p, se = stats.linregress(ydata_ind, xdata_ind)



x=np.linspace(-0.2, 0.45, 100)

ax[0].plot(x, x*slope +intercept, color='k', linestyle=':', label='')

#x = np.linspace(min(xdata), max(xdata), 200)
#reg = LinearRegression(fit_intercept=True).fit([[j] for j in xdata], [[j] for j in ydata])
#plt.plot(x, x * reg.coef_[0][0] + reg.intercept_[0], ':', color="k", alpha=0.7, linewidth=1, zorder=1)

ax[0].text(-0.15,0.25, 'R = ' + str("%.3f" % r) + '\n s = ' + str("%.3f" % slope)+ '\n p = ' + str("%.3f" % p)  , horizontalalignment='left',fontsize=15,color='black')

ax[0].axhline(y=0, color='grey', linestyle='-')    
    
ax[0].set_ylabel(r"Surface temperature change due to HCs /K", fontsize=15)
ax[0].set_xlabel(r"piClim ERF of HCs /W m$^{-2}$", fontsize=15)    


medianprops = dict(linestyle='-', linewidth=1.5, color='k')
box_plot=ax[1].boxplot(xdata_ind, showmeans=True, widths=0.4, meanprops={"markerfacecolor":"silver",  "markeredgecolor":"k","markersize":"10", "markeredgewidth":'1'}, medianprops=medianprops)

ax[1].set_xticks([])

scale=[8,1]    
for n, axis in enumerate(ax):
        axis.text(0.1/scale[n],0.94, "("+string.ascii_lowercase[n]+")", transform=axis.transAxes, size=15)        
        
        
for i in range(len(ax)):
    
    for tick in ax[i].get_xticklabels():
        tick.set_fontsize(15)
    for tick in ax[i].get_yticklabels():
        tick.set_fontsize(15)
   

plt.savefig("hc_erf_piclim_vs_coupledsim_tas_change_duetoHCs.pdf", bbox_inches="tight")
plt.savefig("hc_erf_piclim_vs_coupledsim_tas_change_duetoHCs.png", bbox_inches="tight")

plt.savefig("hc_erf_piclim_vs_coupledsim_tas_change_duetoHCs.svg", bbox_inches="tight", facecolor="white")


#2 problems with uncertainty calculation here: 
#1. interannual variability not taken into account, 
#2. sometimes only 3 data points used to determine standard deviation

In [ ]:
from scipy import stats
import scipy
import string


markers = ["o", "^", "s", "*", "P", "X", "p"]
colors = ["royalblue", "peru", "forestgreen", "firebrick", "purple", "sienna"]


xerror = np.sqrt(np.array([np.var(cesm_global_indiv_warming) + np.var(cesm_hc_global_indiv_warming),
         np.var(cnrm_global_indiv_warming) + np.var(cnrm_hc_global_indiv_warming),
         np.var(gfdl_global_indiv_warming) + np.var(gfdl_hc_global_indiv_warming),
         np.var(giss_global_indiv_warming) + np.var(giss_hc_global_indiv_warming),
         np.var(mri_global_indiv_warming) + np.var(mri_hc_global_indiv_warming),
         np.nanvar(ukesm_global_indiv_warming) + np.nanvar(ukesm_hc_global_indiv_warming)]))



xdata_ind=[]
ydata_ind=[]
colors_ind=[]
x_data=[]

i=0
 
for j in range(len(cesm_hc_global_indiv_warming)):
    if j<3: xdata_ind.append(cesm_global_indiv_warming[j] - cesm_hc_global_indiv_warming[j])
    if j==3: xdata_ind.append(cesm_global_indiv_warming[0] - cesm_hc_global_indiv_warming[j])
    ydata_ind.append(mean_global_erf_hc_all[0])
    colors_ind.append(colors[0])
    if i==0: start_cesm=i
    i=i+1


    
for j in range(len(cnrm_hc_global_indiv_warming)):
    xdata_ind.append(cnrm_global_indiv_warming[j] - cnrm_hc_global_indiv_warming[j])
    ydata_ind.append(mean_global_erf_hc_all[1])
    colors_ind.append(colors[1])
    if j==0: start_cnrm=i
    i=i+1
    
for j in range(len(gfdl_hc_global_indiv_warming)):
    xdata_ind.append(gfdl_global_indiv_warming[j] - gfdl_hc_global_indiv_warming[j])
    ydata_ind.append(mean_global_erf_hc_all[2])
    colors_ind.append(colors[2])
    if j==0: start_gfdl=i
    i=i+1
    
for j in range(len(giss_hc_global_indiv_warming)):
    xdata_ind.append(giss_global_indiv_warming[j] - giss_hc_global_indiv_warming[j])
    ydata_ind.append(mean_global_erf_hc_all[3])
    colors_ind.append(colors[3])
    if j==0: start_giss=i
    i=i+1
    
numbers_mri=[1,3,5,2,4,6]
for j in range(len(mri_hc_global_indiv_warming)):
    xdata_ind.append(mri_global_indiv_warming[numbers_mri[j]] - mri_hc_global_indiv_warming[j])
    ydata_ind.append(mean_global_erf_hc_all[4])
    colors_ind.append(colors[4])
    if j==0: start_mri=i
    i=i+1
    
for j in range(len(ukesm_hc_global_indiv_warming)):
    xdata_ind.append(ukesm_global_indiv_warming[j] - ukesm_hc_global_indiv_warming[j])
    ydata_ind.append(mean_global_erf_hc_all[5])
    colors_ind.append(colors[5])
    if j==0: start_ukesm=i
    i=i+1
   
xdata_err_ind=[]
xdata_err_ind.append([np.min(xdata_ind[0:len(cesm_hc_global_indiv_warming)]-np.mean(xdata_ind[0:len(cesm_hc_global_indiv_warming)])),np.max(xdata_ind[0:len(cesm_hc_global_indiv_warming)]-np.mean(xdata_ind[0:len(cesm_hc_global_indiv_warming)]))] )
xdata_err_ind.append([np.min(xdata_ind[start_cnrm:start_gfdl]-np.mean(xdata_ind[start_cnrm:start_gfdl])),np.max(xdata_ind[start_cnrm:start_gfdl]-np.mean(xdata_ind[start_cnrm:start_gfdl]))] )
xdata_err_ind.append([np.min(xdata_ind[start_gfdl:start_giss]-np.mean(xdata_ind[start_gfdl:start_giss])),np.max(xdata_ind[start_gfdl:start_giss]-np.mean(xdata_ind[start_gfdl:start_giss]))] )
xdata_err_ind.append([np.min(xdata_ind[start_giss:start_mri]-np.mean(xdata_ind[start_giss:start_mri])),np.max(xdata_ind[start_giss:start_mri]-np.mean(xdata_ind[start_giss:start_mri]))] )
xdata_err_ind.append([np.min(xdata_ind[start_mri:start_ukesm]-np.mean(xdata_ind[start_mri:start_ukesm])),np.max(xdata_ind[start_mri:start_ukesm]-np.mean(xdata_ind[start_mri:start_ukesm]))] )
xdata_err_ind.append([np.min(xdata_ind[start_ukesm:len(xdata_ind)]-np.mean(xdata_ind[start_ukesm:len(xdata_ind)])),np.max(xdata_ind[start_ukesm:len(xdata_ind)]-np.mean(xdata_ind[start_ukesm:len(xdata_ind)]))] )

x_data=[]

x_data.append(np.mean(xdata_ind[0:len(cesm_hc_global_indiv_warming)]))
x_data.append(np.mean(xdata_ind[start_cnrm:start_gfdl]))
x_data.append(np.mean(xdata_ind[start_gfdl:start_giss]))
x_data.append(np.mean(xdata_ind[start_giss:start_mri]))
x_data.append(np.mean(xdata_ind[start_mri:start_ukesm]))
x_data.append(np.mean(xdata_ind[start_ukesm:len(xdata_ind)]))


t_cesm, p_cesm = scipy.stats.ttest_ind(cesm_global_indiv_warming,cesm_hc_global_indiv_warming,  equal_var=False)
t_cnrm, p_cnrm = scipy.stats.ttest_ind(cnrm_global_all_warming,cnrm_hc_global_all_warming,  equal_var=False)
t_gfdl, p_gfdl= scipy.stats.ttest_ind(gfdl_global_indiv_warming,gfdl_hc_global_indiv_warming,  equal_var=False)
t_giss, p_giss = scipy.stats.ttest_ind(giss_global_indiv_warming,giss_hc_global_indiv_warming,  equal_var=False)
t_mri, p_mri = scipy.stats.ttest_ind(mri_global_indiv_warming,mri_hc_global_indiv_warming,  equal_var=False)
t_ukesm, p_ukesm = scipy.stats.ttest_ind(ukesm_global_indiv_warming, ukesm_hc_global_indiv_warming,  equal_var=False)

#print(p_cesm)
#print(p_cnrm)
#print(p_gfdl)
#print(p_giss)
#print(p_mri)
#print(p_ukesm)



ydata = mean_global_erf_hc_all

yerror = stdev_global_erf_hc_all
                 
fig, ax = plt.subplots(1,2, figsize=(7.5,5), constrained_layout=True, sharex='col', sharey='row', gridspec_kw={'width_ratios': [8,1]})

#ax[0] = plt.gca()
for i in range(len(x_data)):
    ax[0].errorbar(ydata[i], x_data[i], xerr=yerror[i], yerr=np.reshape(np.abs(xdata_err_ind[i]),(2,1)), marker=markers[i],
                 markersize=8, zorder=2, capsize = 3, capthick=1, elinewidth=1, color=colors[i], alpha=0.7)
    
    
ax[0].legend(names, fontsize=15, loc='lower right')
    
for i in range(len(xdata_ind)):    
    ax[0].plot(ydata_ind[i], xdata_ind[i], linestyle='', marker='o', color=colors_ind[i], markersize=4, label='')
    
    
slope, intercept, r, p, se = stats.linregress(ydata_ind, xdata_ind)



x=np.linspace(-0.2, 0.45, 100)

ax[0].plot(x, x*slope +intercept, color='k', linestyle=':', label='')

#x = np.linspace(min(xdata), max(xdata), 200)
#reg = LinearRegression(fit_intercept=True).fit([[j] for j in xdata], [[j] for j in ydata])
#plt.plot(x, x * reg.coef_[0][0] + reg.intercept_[0], ':', color="k", alpha=0.7, linewidth=1, zorder=1)

ax[0].text(-0.15,0.25, 'R = ' + str("%.3f" % r) + '\n s = ' + str("%.3f" % slope)+ '\n p = ' + str("%.4f" % p)  , horizontalalignment='left',fontsize=15,color='black')

ax[0].axhline(y=0, color='grey', linestyle='-')    
    
ax[0].set_ylabel(r"Surface temperature change due to HCs /K", fontsize=15)
ax[0].set_xlabel(r"piClim ERF of HCs /W m$^{-2}$", fontsize=15)    


medianprops = dict(linestyle='-', linewidth=1.5, color='k')
box_plot=ax[1].boxplot(xdata_ind, showmeans=True, widths=0.4, meanprops={"markerfacecolor":"silver",  "markeredgecolor":"k","markersize":"10", "markeredgewidth":'1'}, medianprops=medianprops)

ax[1].set_xticks([])

scale=[8,1]    
for n, axis in enumerate(ax):
        axis.text(0.1/scale[n],0.94, "("+string.ascii_lowercase[n]+")", transform=axis.transAxes, size=15)        
        
        
for i in range(len(ax)):
    
    for tick in ax[i].get_xticklabels():
        tick.set_fontsize(15)
    for tick in ax[i].get_yticklabels():
        tick.set_fontsize(15)
   
ax[0].set_ylim(-0.4,0.4)

plt.savefig("hc_erf_piclim_vs_coupledsim_tas_change_duetoHCs.pdf", bbox_inches="tight")
plt.savefig("hc_erf_piclim_vs_coupledsim_tas_change_duetoHCs.png", bbox_inches="tight")

plt.savefig("hc_erf_piclim_vs_coupledsim_tas_change_duetoHCs.svg", bbox_inches="tight", facecolor="white")


#2 problems with uncertainty calculation here: 
#1. interannual variability not taken into account, 
#2. sometimes only 3 data points used to determine standard deviation

## OZONE SARF DATA

In [ ]:
o3_sarf=[-60.0607/1000, -178.9771/1000, -188.2866/1000, -89.7758/1000, -42.976/1000, -327.8389/1000]



clouds=[0.06,-0.05,-0.03,0.03,-0.02,-0.09]


In [ ]:
xdata =o3_sarf
ydata = mean_global_erf_hc_all
yerror = stdev_global_erf_hc_all

colors = ["royalblue", "peru", "forestgreen", "firebrick", "purple", "sienna"]

fig, ax = plt.subplots(1,3,figsize=(12,4), constrained_layout=True, edgecolor='k')


for i in range(len(xdata)):
    ax[0].errorbar(xdata[i], ydata[i],  yerr=yerror[i], marker=markers[i], color=colors[i],
                 markersize=8, zorder=2, capsize = 3, capthick=1, elinewidth=1)

  #  plt.errorbar(xdata[i], ydata[i]-xdata[i],  yerr=yerror[i], marker=markers[i], color='grey',
  #               markersize=8, zorder=2, capsize = 3, capthick=1, elinewidth=1, alpha=0.5)
    

ax[0].legend(names,fontsize=12)
    
x = np.linspace(min(xdata), max(xdata), 200)
slope, intercept, r, p, se = stats.linregress(xdata, ydata)
reg = LinearRegression(fit_intercept=True).fit([[j] for j in xdata], [[j] for j in ydata])
ax[0].plot(x, x*slope +intercept, color='k', linestyle=':', label='')


ax[0].text(max(xdata)-0.25*(max(xdata)-min(xdata)),min(ydata)-0.1*(max(ydata)-min(ydata)), 'R = ' + str("%.3f" % r) + '\n s = ' + str("%.3f" % slope)+ '\n p = ' + str("%.3f" % p)  , horizontalalignment='left',fontsize=12,color='black')

ax[0].set_ylim(-0.22,0.45)

ax[0].set_xlabel("Ozone SARF  /(W m$^{-2}$)", fontsize=12)
ax[0].set_ylabel(r"piClim ERF of HCs /W m$^{-2}$", fontsize=12)   


xdata =clouds
ydata = mean_global_erf_hc_all
yerror = stdev_global_erf_hc_all



for i in range(len(xdata)):
    ax[1].errorbar(xdata[i], ydata[i],  yerr=yerror[i], marker=markers[i], color=colors[i],
                 markersize=8, zorder=2, capsize = 3, capthick=1, elinewidth=1)

  #  plt.errorbar(xdata[i], ydata[i]-xdata[i],  yerr=yerror[i], marker=markers[i], color='grey',
  #               markersize=8, zorder=2, capsize = 3, capthick=1, elinewidth=1, alpha=0.5)
    


x = np.linspace(min(xdata), max(xdata), 200)
slope, intercept, r, p, se = stats.linregress(xdata, ydata)
reg = LinearRegression(fit_intercept=True).fit([[j] for j in xdata], [[j] for j in ydata])
ax[1].plot(x, x*slope +intercept, color='k', linestyle=':', label='')


ax[1].text(max(xdata)-0.25*(max(xdata)-min(xdata)),min(ydata)-0.1*(max(ydata)-min(ydata)), 'R = ' + str("%.3f" % r) + '\n s = ' + str("%.3f" % slope)+ '\n p = ' + str("%.3f" % p)  , horizontalalignment='left',fontsize=12,color='black')


ax[1].set_xlabel("Cloud radiative forcing /(W m$^{-2}$)", fontsize=12)
ax[1].set_ylabel(r"piClim ERF of HCs /W m$^{-2}$", fontsize=12)   

ax[1].set_ylim(-0.22,0.45)

xdata =o3_sarf
ydata = clouds



for i in range(len(xdata)):
    ax[2].plot(xdata[i], ydata[i],  marker=markers[i], color=colors[i],
                 markersize=8)

  #  plt.errorbar(xdata[i], ydata[i]-xdata[i],  yerr=yerror[i], marker=markers[i], color='grey',
  #               markersize=8, zorder=2, capsize = 3, capthick=1, elinewidth=1, alpha=0.5)
    


    
x = np.linspace(min(xdata), max(xdata), 200)
slope, intercept, r, p, se = stats.linregress(xdata, ydata)
reg = LinearRegression(fit_intercept=True).fit([[j] for j in xdata], [[j] for j in ydata])
ax[2].plot(x, x*slope +intercept, color='k', linestyle=':', label='')


ax[2].text(max(xdata)-0.25*(max(xdata)-min(xdata)),min(ydata)+0*(max(ydata)-min(ydata)), 'R = ' + str("%.3f" % r) + '\n s = ' + str("%.3f" % slope)+ '\n p = ' + str("%.3f" % p)  , horizontalalignment='left',fontsize=12,color='black')


ax[2].set_xlabel("Ozone SARF /(W m$^{-2}$)", fontsize=12)
ax[2].set_ylabel(r"Cloud radiative forcing /(W m$^{-2}$)", fontsize=12) 





plt.savefig("O3_SARF_vs_ERF.png", bbox_inches="tight")



In [ ]:
markers = ["o", "^", "s", "*", "P", "X", "p"]
colors = ["royalblue", "peru", "forestgreen", "firebrick", "purple", "sienna"]

xdata = np.array([cesm_o3_histchange_100hpa,
                  cnrm_o3_histchange_100hpa,
                  gfdl_o3_histchange_100hpa,
                  giss_o3_histchange_100hpa,
                  mri_o3_histchange_100hpa,
                  ukesm_o3_histchange_100hpa]) * 10**6

xerror = np.array([np.std(cesm_o3_histchange_100hpa_list),
                   np.std(cnrm_o3_histchange_100hpa_list),
                   np.std(gfdl_o3_histchange_100hpa_list),
                   np.std(giss_o3_histchange_100hpa_list),
                   np.std(mri_o3_histchange_100hpa_list),
                   np.std(ukesm_o3_histchange_100hpa_list)]) * 10**6

ydata = o3_sarf

yerror = 0

cmip6_ref = (np.mean(cmip6_o3_100hpa_yr_means[(1996 - 1850):(2005 - 1850)]) -
             np.mean(cmip6_o3_100hpa_yr_means[(1958 - 1850):(1967 - 1850)])) * 10**6

jra55_ref = (np.mean([i.sel(lv_ISBL1 = 100) for i in jra55_o3_global_mean_yearly[(1996 - 1958):(2005 - 1958)]]) -
             np.mean([i.sel(lv_ISBL1 = 100) for i in jra55_o3_global_mean_yearly[(1958 - 1958):(1967 - 1958)]]))
                 
fig = plt.figure(figsize=(9, 6), dpi=300)
ax = plt.gca()
for i in range(len(xdata)):
    plt.errorbar(xdata[i], ydata[i], xerr=xerror[i],  marker=markers[i],
                 markersize=8, zorder=2, capsize = 3, capthick=1, elinewidth=1, color=colors[i])
    
ax.set_xlim(right=0.01)

plt.legend(names,fontsize=12, loc='upper left')

ymin, ymax = plt.ylim()
plt.plot([cmip6_ref, cmip6_ref], [ymin, ymax], c="darkred", linestyle='--')
plt.text(cmip6_ref - 0.03, ymin + 0.02, 'CMIP6 ozone\nforcing data set', c="darkred", fontsize=12, zorder=1)

plt.plot([jra55_ref, jra55_ref], [ymin, ymax], c="indianred", linestyle='--')
plt.text(jra55_ref + 0.003, ymin + 0.02, 'JRA-55', c="indianred", fontsize=12, zorder=1)

x = np.linspace(min(xdata), 0.01, 200)
slope, intercept, r, p, se = stats.linregress(xdata, ydata)
#reg = LinearRegression(fit_intercept=True).fit([[j] for j in xdata], [[j] for j in ydata])
plt.plot(x, x*slope +intercept, color='k', linestyle=':', label='')


ax.text(-0.12,-0.04, 'R = ' + str("%.3f" % r) + '\n s = ' + str("%.3f" % slope)+ '\n p = ' + str("%.3f" % p)  , horizontalalignment='left',fontsize=12,color='black')


plt.xlabel(r"Historical ozone mixing ratio change at 100 hPa in coupled simulations /ppm", fontsize=12)
plt.ylabel(r"piClim O3 SARF /W m$^{-2}$", fontsize=12)    

plt.savefig("o3_sarf_vs_coupledsim_ozone_depletion_historical_100hPa.png", bbox_inches="tight")

plt.savefig("o3_sarf_vs_coupledsim_ozone_depletion_historical_100hPa.svg", bbox_inches="tight", facecolor="white")